# 🧬 TCGA Pan-Cancer Multi-Modal Classifier
### All 33 Cancer Types × 6 Data Modalities — Clean Version

**Before running:**
1. Set runtime to `A100 GPU` → `Runtime > Change runtime type`
2. Run Section 1 (mount Drive) first and make sure it succeeds
3. Run each section in order — each one saves progress to Drive so you can resume if disconnected

**Estimated times:**
- Section 2 (Download): 4–8 hrs
- Section 3 (Preprocess): 1–2 hrs  
- Section 4 (Train): 2–4 hrs
- Section 5 (Evaluate): 30 min

---
## ⚙️ Section 1 — Setup

In [ ]:
# ── 1a. Install packages ──────────────────────────────────────────────────────
!pip install -q umap-learn pyarrow fastparquet

import os, sys, json, gzip, io, time, warnings, pickle
import requests
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')
print('✅ Packages ready')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 83.2 MB/s eta 0:00:00
✅ Packages ready


In [ ]:
# ── 1b. Mount Google Drive ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/tcga_pancancer')
RAW_DIR  = BASE_DIR / 'raw'
PROC_DIR = BASE_DIR / 'processed'
MDL_DIR  = BASE_DIR / 'models'
RES_DIR  = BASE_DIR / 'results'

for d in [RAW_DIR/'rna', RAW_DIR/'meth', RAW_DIR/'cnv',
          RAW_DIR/'mirna', RAW_DIR/'mut', RAW_DIR/'clinical',
          PROC_DIR, MDL_DIR, RES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Verify Drive is writable
test = BASE_DIR / '.write_test'
test.write_text('ok')
assert test.read_text() == 'ok'
test.unlink()

# Check available space
import shutil
_, used, free = shutil.disk_usage('/content/drive/MyDrive')
print(f'✅ Drive mounted and writable')
print(f'   Used: {used/1e9:.1f} GB  |  Free: {free/1e9:.1f} GB')

Mounted at /content/drive
✅ Drive mounted and writable
   Used: 57.0 GB  |  Free: 196.0 GB


In [ ]:
# ── 1c. Configuration ─────────────────────────────────────────────────────────
CANCER_TYPES = [
    'TCGA-ACC',  'TCGA-BLCA', 'TCGA-BRCA', 'TCGA-CESC', 'TCGA-CHOL',
    'TCGA-COAD', 'TCGA-DLBC', 'TCGA-ESCA', 'TCGA-GBM',  'TCGA-HNSC',
    'TCGA-KICH', 'TCGA-KIRC', 'TCGA-KIRP', 'TCGA-LAML', 'TCGA-LGG',
    'TCGA-LIHC', 'TCGA-LUAD', 'TCGA-LUSC', 'TCGA-MESO', 'TCGA-OV',
    'TCGA-PAAD', 'TCGA-PCPG', 'TCGA-PRAD', 'TCGA-READ', 'TCGA-SARC',
    'TCGA-SKCM', 'TCGA-STAD', 'TCGA-TGCT', 'TCGA-THCA', 'TCGA-THYM',
    'TCGA-UCEC', 'TCGA-UCS',  'TCGA-UVM',
]
NUM_CLASSES        = len(CANCER_TYPES)
TOP_VARIABLE_GENES = 5000
TOP_CPG_SITES      = 10000
TOP_MIRNA          = 1000
TOP_MUT_GENES      = 500
MODALITY_NAMES     = ['rna', 'meth', 'cnv', 'mirna', 'mut', 'clin']
LATENT_DIMS        = {'rna':256, 'meth':256, 'cnv':128, 'mirna':128, 'mut':128, 'clin':64}
BATCH_SIZE         = 64
PRETRAIN_EPOCHS    = 20
FINETUNE_EPOCHS    = 80
GDC_FILES          = 'https://api.gdc.cancer.gov/files'
GDC_DATA           = 'https://api.gdc.cancer.gov/data'
GDC_CASES          = 'https://api.gdc.cancer.gov/cases'
print(f'✅ Config ready — {NUM_CLASSES} cancer types, {len(MODALITY_NAMES)} modalities')

✅ Config ready — 33 cancer types, 6 modalities


In [ ]:
# ── 1d. Shared download function (used by all modalities) ─────────────────────
def download_file(file_id, dest_path, retries=3):
    dest_path = Path(dest_path)
    if dest_path.exists() and dest_path.stat().st_size > 0:
        return True  # already downloaded
    url = f'{GDC_DATA}/{file_id}'
    for attempt in range(retries):
        try:
            r = requests.get(url, stream=True, timeout=120)
            r.raise_for_status()
            tmp = dest_path.with_suffix(dest_path.suffix + '.tmp')
            with open(tmp, 'wb') as f:
                for chunk in r.iter_content(1024 * 512):
                    f.write(chunk)
            tmp.rename(dest_path)
            return True
        except Exception as e:
            if attempt == retries - 1:
                return False
            time.sleep(2 ** attempt)
    return False


def gdc_query(data_type, workflow_type=None, data_format=None,
              experimental_strategy=None, n=15000):
    content = [
        {'op': 'in', 'content': {'field': 'cases.project.project_id', 'value': CANCER_TYPES}},
        {'op': '=',  'content': {'field': 'data_type', 'value': data_type}},
    ]
    if workflow_type:
        content.append({'op': '=', 'content': {'field': 'analysis.workflow_type', 'value': workflow_type}})
    if data_format:
        content.append({'op': '=', 'content': {'field': 'data_format', 'value': data_format}})
    if experimental_strategy:
        content.append({'op': '=', 'content': {'field': 'experimental_strategy', 'value': experimental_strategy}})
    params = {
        'filters': json.dumps({'op': 'and', 'content': content}),
        'fields':  'file_id,file_name,cases.project.project_id,cases.case_id',
        'format':  'JSON', 'size': str(n),
    }
    r = requests.get(GDC_FILES, params=params, timeout=60)
    r.raise_for_status()
    records = []
    for h in r.json()['data']['hits']:
        records.append({
            'file_id':    h['file_id'],
            'file_name':  h['file_name'],
            'case_id':    h['cases'][0]['case_id']                if h.get('cases') else None,
            'project_id': h['cases'][0]['project']['project_id']  if h.get('cases') else None,
        })
    return records


def download_modality(files, dest_dir, ext, workers=8):
    ok = fail = 0
    def _dl(rec):
        cid  = rec.get('case_id') or rec.get('project_id') or rec['file_id']
        dest = Path(dest_dir) / f"{cid}__{rec['file_id']}{ext}"
        return download_file(rec['file_id'], dest), rec
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futures = {ex.submit(_dl, r): r for r in files}
        for fut in tqdm(as_completed(futures), total=len(files)):
            success, _ = fut.result()
            if success: ok += 1
            else: fail += 1
    print(f'  ✅ {ok} ok  |  ❌ {fail} failed')
    return ok, fail

print('✅ Download helpers ready')

✅ Download helpers ready


---
## 📥 Section 2 — Download All Modalities
Each cell is **resumable** — re-running skips already-downloaded files.
Check Drive after each cell to confirm files are appearing before moving on.

In [ ]:
# ── 2a. RNA-seq ───────────────────────────────────────────────────────────────
print('Querying GDC for RNA-seq files...')
rna_files = gdc_query(
    data_type='Gene Expression Quantification',
    workflow_type='STAR - Counts',
    data_format='TSV',
    experimental_strategy='RNA-Seq',
)
print(f'Found {len(rna_files)} files')
pd.DataFrame(rna_files).to_csv(PROC_DIR/'manifest_rna.csv', index=False)
download_modality(rna_files, RAW_DIR/'rna', '.tsv.gz')

Querying GDC for RNA-seq files...
Found 11505 files


  0%|          | 0/11505 [00:00<?, ?it/s]

  ✅ 11505 ok  |  ❌ 0 failed


(11505, 0)

In [ ]:
# ── 2b. DNA Methylation ───────────────────────────────────────────────────────
# NOTE: GDC now uses SeSAMe workflow and TXT format (not TSV)
print('Querying GDC for methylation files...')
meth_files = gdc_query(
    data_type='Methylation Beta Value',
    workflow_type='SeSAMe Methylation Beta Estimation',
    data_format='TXT',
)
print(f'Found {len(meth_files)} files')
pd.DataFrame(meth_files).to_csv(PROC_DIR/'manifest_meth.csv', index=False)
download_modality(meth_files, RAW_DIR/'meth', '.txt')

Querying GDC for methylation files...
Found 12527 files


  0%|          | 0/12527 [00:00<?, ?it/s]

  ✅ 12527 ok  |  ❌ 0 failed


(12527, 0)

In [ ]:
# ── 2c. Copy Number Variation ─────────────────────────────────────────────────
print('Querying GDC for CNV files...')
cnv_files = gdc_query(
    data_type='Copy Number Segment',
    workflow_type='DNAcopy',
)
print(f'Found {len(cnv_files)} files')
pd.DataFrame(cnv_files).to_csv(PROC_DIR/'manifest_cnv.csv', index=False)
download_modality(cnv_files, RAW_DIR/'cnv', '.tsv')

Querying GDC for CNV files...
Found 15000 files


  0%|          | 0/15000 [00:00<?, ?it/s]

  ✅ 15000 ok  |  ❌ 0 failed


(15000, 0)

In [ ]:
# ── 2d. miRNA ─────────────────────────────────────────────────────────────────
print('Querying GDC for miRNA files...')
mirna_files = gdc_query(
    data_type='Isoform Expression Quantification',
    experimental_strategy='miRNA-Seq',
)
print(f'Found {len(mirna_files)} files')
pd.DataFrame(mirna_files).to_csv(PROC_DIR/'manifest_mirna.csv', index=False)
download_modality(mirna_files, RAW_DIR/'mirna', '.tsv')

Querying GDC for miRNA files...
Found 11441 files


  0%|          | 0/11441 [00:00<?, ?it/s]

  ✅ 11441 ok  |  ❌ 0 failed


(11441, 0)

In [ ]:
# ── 2e. Somatic Mutations (MAF) ───────────────────────────────────────────────
print('Querying GDC for mutation MAF files...')
mut_files = gdc_query(
    data_type='Masked Somatic Mutation',
    data_format='MAF',
    n=500,
)
print(f'Found {len(mut_files)} files')
pd.DataFrame(mut_files).to_csv(PROC_DIR/'manifest_mut.csv', index=False)
download_modality(mut_files, RAW_DIR/'mut', '.maf.gz')

Querying GDC for mutation MAF files...
Found 500 files


  0%|          | 0/500 [00:00<?, ?it/s]

  ✅ 500 ok  |  ❌ 0 failed


(500, 0)

In [ ]:
import requests, json, time
from pathlib import Path
from tqdm.notebook import tqdm
import pandas as pd

BASE_DIR = Path('/content/drive/MyDrive/tcga_pancancer')
mut_dir  = BASE_DIR / 'raw/mut'
mut_dir.mkdir(exist_ok=True)

GDC_FILES = 'https://api.gdc.cancer.gov/files'
GDC_DATA  = 'https://api.gdc.cancer.gov/data'

CANCER_TYPES = [
    'TCGA-ACC',  'TCGA-BLCA', 'TCGA-BRCA', 'TCGA-CESC', 'TCGA-CHOL',
    'TCGA-COAD', 'TCGA-DLBC', 'TCGA-ESCA', 'TCGA-GBM',  'TCGA-HNSC',
    'TCGA-KICH', 'TCGA-KIRC', 'TCGA-KIRP', 'TCGA-LAML', 'TCGA-LGG',
    'TCGA-LIHC', 'TCGA-LUAD', 'TCGA-LUSC', 'TCGA-MESO', 'TCGA-OV',
    'TCGA-PAAD', 'TCGA-PCPG', 'TCGA-PRAD', 'TCGA-READ', 'TCGA-SARC',
    'TCGA-SKCM', 'TCGA-STAD', 'TCGA-TGCT', 'TCGA-THCA', 'TCGA-THYM',
    'TCGA-UCEC', 'TCGA-UCS',  'TCGA-UVM'
]

def download_file(file_id, dest_path, retries=3):
    dest_path = Path(dest_path)
    if dest_path.exists() and dest_path.stat().st_size > 1000:
        return True
    url = f'{GDC_DATA}/{file_id}'
    for attempt in range(retries):
        try:
            r = requests.get(url, stream=True, timeout=120)
            r.raise_for_status()
            tmp = dest_path.with_suffix(dest_path.suffix + '.tmp')
            with open(tmp, 'wb') as f:
                for chunk in r.iter_content(1024 * 512):
                    f.write(chunk)
            tmp.rename(dest_path)
            return True
        except Exception as e:
            if attempt == retries - 1:
                return False
            time.sleep(2 ** attempt)
    return False

# Query each cancer type individually to ensure all 33 are represented
print('Querying MAF files per cancer type...')
all_mut_files = []

for project in tqdm(CANCER_TYPES):
    filters = {
        'op': 'and',
        'content': [
            {'op': '=', 'content': {'field': 'cases.project.project_id', 'value': project}},
            {'op': '=', 'content': {'field': 'data_type',   'value': 'Masked Somatic Mutation'}},
            {'op': '=', 'content': {'field': 'data_format', 'value': 'MAF'}},
        ]
    }
    params = {
        'filters': json.dumps(filters),
        'fields':  'file_id,file_name,cases.project.project_id,cases.case_id',
        'format':  'JSON',
        'size':    '500'   # up to 500 per cancer type
    }
    try:
        r = requests.get(GDC_FILES, params=params, timeout=60)
        hits = r.json()['data']['hits']
        for h in hits:
            all_mut_files.append({
                'file_id':    h['file_id'],
                'file_name':  h['file_name'],
                'case_id':    h['cases'][0]['case_id']                if h.get('cases') else None,
                'project_id': h['cases'][0]['project']['project_id']  if h.get('cases') else None,
            })
    except Exception as e:
        print(f'  Warning {project}: {e}')

print(f'\nTotal MAF files found: {len(all_mut_files)}')
df = pd.DataFrame(all_mut_files)
print(df['project_id'].value_counts().to_string())

# Save updated manifest
df.to_csv(BASE_DIR/'processed'/'manifest_mut.csv', index=False)

# Download missing files only
print('\nDownloading missing MAF files...')
ok = fail = 0
for rec in tqdm(all_mut_files):
    pid  = rec.get('project_id') or 'UNKNOWN'
    dest = mut_dir / f"{pid}__{rec['file_id']}.maf.gz"
    if download_file(rec['file_id'], dest):
        ok += 1
    else:
        fail += 1

print(f'✅ Done: {ok} ok  |  ❌ {fail} failed')

Querying MAF files per cancer type...


  0%|          | 0/33 [00:00<?, ?it/s]


Total MAF files found: 9920
project_id
TCGA-BRCA    500
TCGA-PRAD    500
TCGA-LUAD    500
TCGA-LGG     500
TCGA-HNSC    500
TCGA-LUSC    500
TCGA-UCEC    500
TCGA-THCA    498
TCGA-OV      482
TCGA-SKCM    472
TCGA-GBM     464
TCGA-COAD    457
TCGA-STAD    434
TCGA-BLCA    415
TCGA-KIRC    412
TCGA-LIHC    372
TCGA-CESC    289
TCGA-KIRP    283
TCGA-SARC    238
TCGA-ESCA    185
TCGA-PCPG    184
TCGA-PAAD    180
TCGA-READ    161
TCGA-LAML    153
TCGA-TGCT    147
TCGA-THYM    123
TCGA-ACC      90
TCGA-UVM      80
TCGA-MESO     80
TCGA-KICH     66
TCGA-UCS      57
TCGA-CHOL     51
TCGA-DLBC     47



  0%|          | 0/9920 [00:00<?, ?it/s]

✅ Done: 9920 ok  |  ❌ 0 failed


In [ ]:
# ── 2f. Clinical data (from GDC cases API) ────────────────────────────────────
print('Fetching clinical data...')
clinical_records = []
for project in tqdm(CANCER_TYPES):
    filters = {'op': 'in', 'content': {'field': 'project.project_id', 'value': [project]}}
    params  = {
        'filters': json.dumps(filters),
        'fields':  ('case_id,diagnoses.age_at_diagnosis,diagnoses.tumor_stage,'
                    'diagnoses.vital_status,diagnoses.days_to_death,'
                    'demographic.gender,demographic.race,'
                    'exposures.pack_years_smoked,project.project_id'),
        'format':  'JSON', 'size': '2000',
    }
    try:
        r = requests.get(GDC_CASES, params=params, timeout=60)
        for h in r.json()['data']['hits']:
            diag = h.get('diagnoses', [{}])[0]
            expo = h.get('exposures', [{}])[0]
            clinical_records.append({
                'case_id':           h.get('case_id'),
                'project_id':        project,
                'gender':            h.get('demographic', {}).get('gender'),
                'race':              h.get('demographic', {}).get('race'),
                'age_at_diagnosis':  diag.get('age_at_diagnosis'),
                'tumor_stage':       diag.get('tumor_stage'),
                'vital_status':      diag.get('vital_status'),
                'days_to_death':     diag.get('days_to_death'),
                'pack_years_smoked': expo.get('pack_years_smoked'),
            })
    except Exception as e:
        print(f'  Warning {project}: {e}')

clin_df = pd.DataFrame(clinical_records)
clin_df.to_parquet(RAW_DIR/'clinical'/'clinical_raw.parquet', index=False)
print(f'✅ Clinical data: {len(clin_df)} records saved')

Fetching clinical data...


  0%|          | 0/33 [00:00<?, ?it/s]

✅ Clinical data: 11428 records saved


In [ ]:
# ── 2g. Download summary ──────────────────────────────────────────────────────
print('Download summary:')
total_size = 0
for mod in ['rna', 'meth', 'cnv', 'mirna', 'mut']:
    folder = RAW_DIR / mod
    files  = [f for f in folder.glob('*') if not f.suffix == '.tmp']
    size   = sum(f.stat().st_size for f in files) / 1e9
    total_size += size
    print(f'  {mod:6s}: {len(files):5d} files  {size:.1f} GB')
print(f'  Total:  {total_size:.1f} GB')

Download summary:
  rna   : 11505 files  48.7 GB
  meth  : 12527 files  131.9 GB
  cnv   : 15000 files  0.6 GB
  mirna : 11441 files  4.3 GB
  mut   : 10420 files  0.8 GB
  Total:  186.4 GB


---
## 🔧 Section 3 — Preprocessing
Parses raw files into feature matrices. Run one modality at a time.
Each saves a `.parquet` file to Drive — these are all you need for training.

In [ ]:
import gzip
from pathlib import Path

mut_dir = Path('/content/drive/MyDrive/tcga_pancancer/raw/mut')
f0 = list(mut_dir.glob('*.maf.gz'))[0]
print(f'File: {f0.name}')

try:
    with gzip.open(f0, 'rt') as f:
        for i, line in enumerate(f):
            print(line.strip()[:100])
            if i > 3: break
    print('✅ Actually gzipped')
except Exception as e:
    print(f'gzip failed: {e}')
    print('Opening as plain text...')
    with open(f0, 'r', errors='replace') as f:
        for i, line in enumerate(f):
            print(line.strip()[:100])
            if i > 3: break
    print('⚠️ Plain text despite .gz extension')

File: TCGA-THCA__446cbace-4f29-4c03-b7f1-7967f614c8e0.maf.gz
#version gdc-1.0.0
#annotation.spec gdc-2.0.0-aliquot-merged-masked
#contigs chr1,chr2,chr3,chr4,chr5,chr6,chr7,chr8,chr9,chr10,chr11,chr12,chr13,chr14,chr15,chr16,chr1
#sort.order BarcodesAndCoordinate
#filedate 20220517
✅ Actually gzipped


In [ ]:
# ── 3a. RNA-seq → parquet ─────────────────────────────────────────────────────
# Skip if already done
if (PROC_DIR/'rna_matrix.parquet').exists():
    print('✅ RNA matrix already exists, skipping')
    rna_df = pd.read_parquet(PROC_DIR/'rna_matrix.parquet')
    print(f'   Shape: {rna_df.shape}')
else:
    # ... rest of RNA parsing code
    from sklearn.preprocessing import StandardScaler

    print('Parsing RNA-seq files...')
    manifest = pd.read_csv(PROC_DIR/'manifest_rna.csv')
    rows, labels, case_ids, gene_index = [], [], [], None

    for _, row in tqdm(manifest.iterrows(), total=len(manifest)):
        path = RAW_DIR/'rna'/f"{row['case_id']}__{row['file_id']}.tsv.gz"
        if not path.exists(): continue
        try:
            with open(path, 'r') as f:
                df = pd.read_csv(f, sep='\t', comment='#')
            if 'tpm_unstranded' not in df.columns: continue
            mask_g = df['gene_id'].str.startswith('ENSG')
            sub    = df[mask_g].copy()
            sub['gene_id'] = sub['gene_id'].str.split('.').str[0]
            s = sub.set_index('gene_id')['tpm_unstranded'].astype(np.float32)
            if gene_index is None: gene_index = s.index
            else: s = s.reindex(gene_index, fill_value=0)
            rows.append(s.values)
            labels.append(row['project_id'])
            case_ids.append(row['case_id'])
        except Exception: continue

    print(f'Parsed {len(rows)} samples')
    X = np.log1p(np.array(rows, dtype=np.float32))
    top_idx = np.argsort(X.var(axis=0))[::-1][:TOP_VARIABLE_GENES]
    X = StandardScaler().fit_transform(X[:, top_idx]).astype(np.float32)
    rna_df = pd.DataFrame(X, index=case_ids, columns=gene_index[top_idx])
    rna_df.index.name = 'case_id'
    rna_df.to_parquet(PROC_DIR/'rna_matrix.parquet')
    pd.Series(labels, index=case_ids, name='project_id').to_csv(PROC_DIR/'labels_rna.csv')
    print(f'✅ RNA matrix: {rna_df.shape}')

✅ RNA matrix already exists, skipping
   Shape: (11505, 5000)


In [ ]:
# ── 3b. Methylation → parquet ─────────────────────────────────────────────────
# NOTE: New SeSAMe format — files are .txt not .tsv.gz
# Skip if already done
if (PROC_DIR/'meth_matrix.parquet').exists():
    print('✅ Methylation matrix already exists, skipping')
    meth_df = pd.read_parquet(PROC_DIR/'meth_matrix.parquet')
    print(f'   Shape: {meth_df.shape}')
else:
    # ... rest of methylation parsing code
    print('Parsing methylation files...')
    manifest = pd.read_csv(PROC_DIR/'manifest_meth.csv')
    all_betas, labels, case_ids, cpg_index = [], [], [], None

    # First pass: get CpG index from first valid file
    for _, row in manifest.head(10).iterrows():
        path = RAW_DIR/'meth'/f"{row['case_id']}__{row['file_id']}.txt"
        if not path.exists(): continue
        try:
            df = pd.read_csv(path, sep='\t', header=0)
            # SeSAMe format: first col is probe ID, second is beta value
            df.columns = ['cpg_id', 'beta_value'] + list(df.columns[2:])
            df = df[['cpg_id', 'beta_value']].dropna()
            cpg_index = pd.Index(df['cpg_id'].values)
            print(f'CpG sites: {len(cpg_index):,}')
            print(f'Sample columns: {df.columns.tolist()}')
            break
        except Exception as e:
            print(f'Parse error: {e}')
            continue

    # Second pass: load all files
    for _, row in tqdm(manifest.iterrows(), total=len(manifest)):
        path = RAW_DIR/'meth'/f"{row['case_id']}__{row['file_id']}.txt"
        if not path.exists(): continue
        try:
            df = pd.read_csv(path, sep='\t', header=0)
            df.columns = ['cpg_id', 'beta_value'] + list(df.columns[2:])
            s  = df[['cpg_id','beta_value']].dropna().set_index('cpg_id')['beta_value']
            s  = s.reindex(cpg_index, fill_value=np.nan).astype(np.float32)
            all_betas.append(s.values)
            labels.append(row['project_id'])
            case_ids.append(row['case_id'])
        except Exception: continue

    M = np.array(all_betas, dtype=np.float32)
    var = np.nanvar(M, axis=0)
    top_idx = np.argsort(var)[::-1][:TOP_CPG_SITES]
    M = M[:, top_idx]
    col_med = np.nanmedian(M, axis=0)
    nan_mask = np.isnan(M)
    M[nan_mask] = np.take(col_med, np.where(nan_mask)[1])
    M = np.clip(M, 0, 1)
    M = StandardScaler().fit_transform(M).astype(np.float32)
    meth_df = pd.DataFrame(M, index=case_ids, columns=cpg_index[top_idx])
    meth_df.index.name = 'case_id'
    meth_df.to_parquet(PROC_DIR/'meth_matrix.parquet')
    pd.Series(labels, index=case_ids, name='project_id').to_csv(PROC_DIR/'labels_meth.csv')
    print(f'✅ Methylation matrix: {meth_df.shape}')

✅ Methylation matrix already exists, skipping
   Shape: (12527, 10000)


In [ ]:
# ── 3c. CNV → parquet ─────────────────────────────────────────────────────────
if (PROC_DIR/'cnv_matrix.parquet').exists():
    print('✅ CNV matrix already exists, skipping')
    cnv_df = pd.read_parquet(PROC_DIR/'cnv_matrix.parquet')
    print(f'   Shape: {cnv_df.shape}')
else:
    # paste rest of 3c cell here
    CHROM_ARMS = {
        '1p':('1',0,125000000),    '1q':('1',125000000,248956422),
        '2p':('2',0,93900000),     '2q':('2',93900000,242193529),
        '3p':('3',0,90900000),     '3q':('3',90900000,198295559),
        '4p':('4',0,50000000),     '4q':('4',50000000,190214555),
        '5p':('5',0,48800000),     '5q':('5',48800000,181538259),
        '6p':('6',0,60600000),     '6q':('6',60600000,170805979),
        '7p':('7',0,59900000),     '7q':('7',59900000,159345973),
        '8p':('8',0,45200000),     '8q':('8',45200000,145138636),
        '9p':('9',0,47300000),     '9q':('9',47300000,138394717),
        '10p':('10',0,38000000),   '10q':('10',38000000,133797422),
        '11p':('11',0,53400000),   '11q':('11',53400000,135086622),
        '12p':('12',0,35800000),   '12q':('12',35800000,133275309),
        '13q':('13',0,114364328),  '14q':('14',0,107043718),
        '15q':('15',0,101991189),  '16p':('16',0,46485900),
        '16q':('16',46485900,90338345), '17p':('17',0,25100000),
        '17q':('17',25100000,83257441), '18p':('18',0,18500000),
        '18q':('18',18500000,80373285), '19p':('19',0,26200000),
        '19q':('19',26200000,58617616), '20p':('20',0,28100000),
        '20q':('20',28100000,64444167), '21q':('21',0,46709983),
        '22q':('22',0,50818468),        'Xp':('X',0,60600000),
        'Xq':('X',60600000,156040895),
    }
    ARM_NAMES = list(CHROM_ARMS.keys())

    print('Parsing CNV files...')
    manifest = pd.read_csv(PROC_DIR/'manifest_cnv.csv')
    rows, labels, case_ids = [], [], []

    for _, row in tqdm(manifest.iterrows(), total=len(manifest)):
        path = RAW_DIR/'cnv'/f"{row['case_id']}__{row['file_id']}.tsv"
        if not path.exists(): continue
        try:
            df = pd.read_csv(path, sep='\t')
            df.columns = [c.strip() for c in df.columns]
            df['Chromosome'] = df['Chromosome'].astype(str).str.replace('chr','',regex=False)
            arm_profile = {}
            for arm, (chrom, start, end) in CHROM_ARMS.items():
                m = ((df['Chromosome']==chrom) & (df['Start']<end) & (df['End']>start))
                vals = df.loc[m, 'Segment_Mean']
                arm_profile[arm] = float(vals.mean()) if len(vals) > 0 else 0.0
            rows.append(list(arm_profile.values()))
            labels.append(row['project_id'])
            case_ids.append(row['case_id'])
        except Exception: continue

    X = np.clip(np.array(rows, dtype=np.float32), -3, 3)
    X = StandardScaler().fit_transform(X).astype(np.float32)
    cnv_df = pd.DataFrame(X, index=case_ids, columns=ARM_NAMES)
    cnv_df.index.name = 'case_id'
    cnv_df.to_parquet(PROC_DIR/'cnv_matrix.parquet')
    pd.Series(labels, index=case_ids, name='project_id').to_csv(PROC_DIR/'labels_cnv.csv')
    print(f'✅ CNV matrix: {cnv_df.shape}')

✅ CNV matrix already exists, skipping
   Shape: (15000, 41)


In [ ]:
# ── 3d. miRNA → parquet ───────────────────────────────────────────────────────
if (PROC_DIR/'mirna_matrix.parquet').exists():
    print('✅ miRNA matrix already exists, skipping')
    mirna_df = pd.read_parquet(PROC_DIR/'mirna_matrix.parquet')
    print(f'   Shape: {mirna_df.shape}')
else:
    # paste rest of 3d cell here
    print('Parsing miRNA files...')
    manifest = pd.read_csv(PROC_DIR/'manifest_mirna.csv')
    rows, labels, case_ids, mirna_index = [], [], [], None

    for _, row in tqdm(manifest.iterrows(), total=len(manifest)):
        path = RAW_DIR/'mirna'/f"{row['case_id']}__{row['file_id']}.tsv"
        if not path.exists(): continue
        try:
            df = pd.read_csv(path, sep='\t')
            if 'miRNA_ID' not in df.columns: continue
            rpm = 'reads_per_million_miRNA_mapped'
            if rpm not in df.columns: continue
            s = df.groupby('miRNA_ID')[rpm].sum().astype(np.float32)
            if mirna_index is None: mirna_index = s.index
            else: s = s.reindex(mirna_index, fill_value=0)
            rows.append(s.values)
            labels.append(row['project_id'])
            case_ids.append(row['case_id'])
        except Exception: continue

    X = np.log1p(np.array(rows, dtype=np.float32))
    top_idx = np.argsort(X.var(axis=0))[::-1][:TOP_MIRNA]
    X = StandardScaler().fit_transform(X[:, top_idx]).astype(np.float32)
    mirna_df = pd.DataFrame(X, index=case_ids, columns=mirna_index[top_idx])
    mirna_df.index.name = 'case_id'
    mirna_df.to_parquet(PROC_DIR/'mirna_matrix.parquet')
    pd.Series(labels, index=case_ids, name='project_id').to_csv(PROC_DIR/'labels_mirna.csv')
    print(f'✅ miRNA matrix: {mirna_df.shape}')

✅ miRNA matrix already exists, skipping
   Shape: (11441, 508)


In [ ]:
# ── 3e. Mutations → parquet ───────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
if (PROC_DIR/'mut_matrix.parquet').exists():
    print('✅ Mutation matrix already exists, skipping')
    mut_df = pd.read_parquet(PROC_DIR/'mut_matrix.parquet')
    print(f'   Shape: {mut_df.shape}')
else:
    # paste rest of 3e cell here
    print('Parsing MAF files...')
    manifest = pd.read_csv(PROC_DIR/'manifest_mut.csv')
    all_mafs = []

    for _, row in tqdm(manifest.iterrows(), total=len(manifest)):
        pid  = row.get('project_id') or 'UNKNOWN'
        path = RAW_DIR/'mut'/f"{pid}__{row['file_id']}.maf.gz"
        if not path.exists(): continue
        try:
            with gzip.open(path, 'rt') as f:
                lines = [l for l in f if not l.startswith('#')]
            df = pd.read_csv(io.StringIO(''.join(lines)), sep='\t', low_memory=False)
            keep = ['Hugo_Symbol','Variant_Type','Tumor_Sample_Barcode']
            df = df[[c for c in keep if c in df.columns]]
            df['project_id'] = pid
            all_mafs.append(df)
        except Exception: continue

    maf_all = pd.concat(all_mafs, ignore_index=True)
    maf_all['case_id'] = maf_all['Tumor_Sample_Barcode'].str[:12]
    top_genes = maf_all['Hugo_Symbol'].value_counts().head(TOP_MUT_GENES).index.tolist()
    mut_pivot = (maf_all[maf_all['Hugo_Symbol'].isin(top_genes)]
                .groupby(['case_id','Hugo_Symbol']).size()
                .unstack(fill_value=0).clip(upper=1).astype(np.float32))
    mut_pivot = mut_pivot.reindex(columns=top_genes, fill_value=0)
    tmb = maf_all.groupby('case_id').size().rename('log_tmb').apply(np.log1p)
    mut_pivot = mut_pivot.join(tmb, how='left').fillna(0)
    X = StandardScaler().fit_transform(mut_pivot.values).astype(np.float32)
    mut_df = pd.DataFrame(X, index=mut_pivot.index, columns=mut_pivot.columns)
    mut_df.index.name = 'case_id'
    mut_df.to_parquet(PROC_DIR/'mut_matrix.parquet')
    maf_all.groupby('case_id')['project_id'].first().reindex(mut_pivot.index).to_csv(PROC_DIR/'labels_mut.csv')
    print(f'✅ Mutation matrix: {mut_df.shape}')

✅ Mutation matrix already exists, skipping
   Shape: (9106, 501)


In [ ]:
# ── 3f. Clinical → parquet ────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
import pandas as pd
import numpy as np
import pickle

print('Processing clinical data...')
df = pd.read_parquet(RAW_DIR/'clinical'/'clinical_raw.parquet').set_index('case_id')

# Convert None to NaN explicitly
df = df.where(df.notna(), other=np.nan)
df = df.replace({None: np.nan, 'None': np.nan, '': np.nan})

# Numeric features
df['age_years']         = pd.to_numeric(df['age_at_diagnosis'], errors='coerce') / 365.25
df['days_to_death']     = pd.to_numeric(df['days_to_death'],    errors='coerce')
df['pack_years_smoked'] = pd.to_numeric(df['pack_years_smoked'],errors='coerce')

# Categorical features — fill NaN before encoding
for col in ['gender', 'race', 'tumor_stage', 'vital_status']:
    df[col] = df[col].fillna('unknown').astype(str)
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    df[col+'_enc'] = enc.fit_transform(df[[col]]).astype(np.float32)

CLIN_FEATURES = ['age_years', 'days_to_death', 'pack_years_smoked',
                 'gender_enc', 'race_enc', 'tumor_stage_enc', 'vital_status_enc']

feat = df[CLIN_FEATURES].astype(np.float32)

# Fill remaining NaNs with column median
for col in CLIN_FEATURES:
    median = feat[col].median()
    if np.isnan(median):
        median = 0.0
    feat[col] = feat[col].fillna(median)

# Final check
assert feat.isna().sum().sum() == 0, 'Still has NaNs!'
assert not np.isinf(feat.values).any(), 'Has Infs!'

X = StandardScaler().fit_transform(feat.values).astype(np.float32)
clin_df = pd.DataFrame(X, index=feat.index, columns=CLIN_FEATURES)
clin_df.index.name = 'case_id'
clin_df.to_parquet(PROC_DIR/'clin_matrix.parquet')
df['project_id'].to_csv(PROC_DIR/'labels_clin.csv')

print(f'✅ Clinical matrix: {clin_df.shape}')
print(f'   NaNs: {clin_df.isna().sum().sum()}')
print(f'   Value range: {X.min():.2f} to {X.max():.2f}')

Processing clinical data...
✅ Clinical matrix: (11428, 7)
   NaNs: 0
   Value range: -3.78 to 52.51


In [ ]:
# ── 3g. Build train/val/test splits ──────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import numpy as np
import pickle

if (PROC_DIR/'splits.npz').exists():
    print('✅ Splits already exist, skipping')
else:
    print('Building splits...')
    all_series = []
    for mod in MODALITY_NAMES:
        p = PROC_DIR / f'labels_{mod}.csv'
        if not p.exists(): continue
        df = pd.read_csv(p)
        df.columns = ['case_id', 'project_id']
        s = df.set_index('case_id')['project_id']
        all_series.append(s)

    # Fix — deduplicate by index not by value
    all_labels = pd.concat(all_series)
    all_labels = all_labels[~all_labels.index.duplicated(keep='first')]
    print(f'Total unique case_ids: {len(all_labels)}')

    rna_df    = pd.read_parquet(PROC_DIR/'rna_matrix.parquet')
    valid_ids = [cid for cid in all_labels.index.unique() if cid in rna_df.index]
    labels    = all_labels.reindex(valid_ids)
    print(f'Valid case_ids (have RNA): {len(valid_ids)}')

    # Drop cancer types with fewer than 5 samples
    label_counts = labels.value_counts()
    keep_types   = label_counts[label_counts >= 5].index
    mask         = labels.isin(keep_types)
    valid_ids    = np.array(valid_ids)[mask.values]
    labels       = labels[mask]
    dropped      = set(label_counts[label_counts < 5].index)
    if dropped:
        print(f'Dropped cancer types with < 5 samples: {dropped}')

    le = LabelEncoder()
    y  = le.fit_transform(labels)

    idx = np.arange(len(valid_ids))
    idx_tv,    idx_test  = train_test_split(idx,    test_size=0.15, stratify=y,          random_state=42)
    idx_train, idx_val   = train_test_split(idx_tv, test_size=0.15, stratify=y[idx_tv],  random_state=42)

    np.savez(PROC_DIR/'splits.npz',
             idx_train=idx_train, idx_val=idx_val, idx_test=idx_test,
             y=y, valid_ids=np.array(valid_ids))
    with open(PROC_DIR/'label_encoder.pkl', 'wb') as f:
        pickle.dump(le, f)

    print(f'\n✅ Splits → train={len(idx_train):,}  val={len(idx_val):,}  test={len(idx_test):,}')
    print(f'   Total samples: {len(valid_ids):,}')
    print(f'   Cancer types:  {len(np.unique(y))}')
    print(f'   Classes: {list(le.classes_)}')

✅ Splits already exist, skipping


---
## 🏗️ Section 4 — Model + Training

In [ ]:
# ── 4a. Load data ─────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

splits   = np.load(PROC_DIR/'splits.npz', allow_pickle=True)
idx_train = splits['idx_train']
idx_val   = splits['idx_val']
idx_test  = splits['idx_test']
y         = splits['y']
case_ids  = splits['valid_ids']
with open(PROC_DIR/'label_encoder.pkl','rb') as f:
    le = pickle.load(f)

mod_data, mod_maps, mod_dims = [], [], {}
for mod in MODALITY_NAMES:
    p = PROC_DIR / f'{mod}_matrix.parquet'
    if p.exists():
        df = pd.read_parquet(p)
        mod_maps.append({cid: i for i, cid in enumerate(df.index)})
        mod_data.append(df.values.astype(np.float32))
        mod_dims[mod] = df.shape[1]
        print(f'  {mod:6s}: {df.shape}')
    else:
        mod_maps.append({})
        mod_data.append(np.zeros((1,1), dtype=np.float32))
        mod_dims[mod] = 1
        print(f'  {mod:6s}: NOT FOUND — will be masked out')

print(f'\n✅ Data loaded — {len(case_ids):,} total samples')

Device: cuda
  rna   : (11505, 5000)
  meth  : (12527, 10000)
  cnv   : (15000, 41)
  mirna : (11441, 508)
  mut   : (9106, 501)
  clin  : (11428, 7)

✅ Data loaded — 10,517 total samples


In [ ]:
# ── 4b. Dataset + Model ───────────────────────────────────────────────────────
class MultiModalDataset(Dataset):
    def __init__(self, indices, mod_data, mod_maps, y, case_ids):
        self.indices  = indices
        self.mod_data = mod_data
        self.mod_maps = mod_maps
        self.y        = y
        self.case_ids = case_ids
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        idx   = self.indices[i]
        cid   = self.case_ids[idx]
        tensors, mask = [], []
        for data, mapping in zip(self.mod_data, self.mod_maps):
            if cid in mapping:
                tensors.append(torch.tensor(data[mapping[cid]], dtype=torch.float32))
                mask.append(1.0)
            else:
                tensors.append(torch.zeros(data.shape[1], dtype=torch.float32))
                mask.append(0.0)
        return tensors, torch.tensor(mask, dtype=torch.float32), torch.tensor(int(self.y[idx]), dtype=torch.long)

def collate_fn(batch):
    tensors_list, masks, labels = zip(*batch)
    return ([torch.stack([s[m] for s in tensors_list]) for m in range(len(tensors_list[0]))],
            torch.stack(masks), torch.stack(labels))

def make_encoder(in_dim, latent_dim, hidden):
    layers, prev = [], in_dim
    for h in hidden:
        layers += [nn.Linear(prev,h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(0.2)]
        prev = h
    layers.append(nn.Linear(prev, latent_dim))
    return nn.Sequential(*layers)

def make_decoder(latent_dim, out_dim, hidden):
    layers, prev = [], latent_dim
    for h in hidden:
        layers += [nn.Linear(prev,h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(0.1)]
        prev = h
    layers.append(nn.Linear(prev, out_dim))
    return nn.Sequential(*layers)

class MultiModalAutoencoder(nn.Module):
    def __init__(self, mod_dims, latent_dims, num_classes):
        super().__init__()
        d, l = mod_dims, latent_dims
        self.encoders = nn.ModuleList([
            make_encoder(d['rna'],   l['rna'],   [2048,1024,512]),
            make_encoder(d['meth'],  l['meth'],  [2048,1024,512]),
            make_encoder(d['cnv'],   l['cnv'],   [256,128]),
            make_encoder(d['mirna'], l['mirna'], [512,256]),
            make_encoder(d['mut'],   l['mut'],   [512,256]),
            make_encoder(d['clin'],  l['clin'],  [64,64]),
        ])
        self.decoders = nn.ModuleList([
            make_decoder(l['rna'],   d['rna'],   [512,1024,2048]),
            make_decoder(l['meth'],  d['meth'],  [512,1024,2048]),
            make_decoder(l['cnv'],   d['cnv'],   [128,256]),
            make_decoder(l['mirna'], d['mirna'], [256,512]),
            make_decoder(l['mut'],   d['mut'],   [256,512]),
            make_decoder(l['clin'],  d['clin'],  [64,64]),
        ])
        self.gate = nn.Linear(len(latent_dims), len(latent_dims))
        fusion_dim = sum(latent_dims.values())
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim,1024), nn.LayerNorm(1024), nn.GELU(), nn.Dropout(0.4),
            nn.Linear(1024,512),        nn.LayerNorm(512),  nn.GELU(), nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )
    def encode_all(self, tensors, mask):
        zs   = [enc(x) for enc, x in zip(self.encoders, tensors)]
        gate = torch.sigmoid(self.gate(mask))
        return torch.cat([z * gate[:,i:i+1] for i,z in enumerate(zs)], dim=-1), zs, gate
    def forward(self, tensors, mask):
        fused, zs, gate = self.encode_all(tensors, mask)
        return self.classifier(fused), [dec(z) for dec,z in zip(self.decoders,zs)], gate

model = MultiModalAutoencoder(mod_dims, LATENT_DIMS, NUM_CLASSES).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ Model ready — {n_params:,} parameters')

train_loader = DataLoader(MultiModalDataset(idx_train,mod_data,mod_maps,y,case_ids),
                          batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader   = DataLoader(MultiModalDataset(idx_val,  mod_data,mod_maps,y,case_ids),
                          batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
test_loader  = DataLoader(MultiModalDataset(idx_test, mod_data,mod_maps,y,case_ids),
                          batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
print(f'   Train batches: {len(train_loader)}')

✅ Model ready — 75,876,420 parameters
   Train batches: 119


In [ ]:
# ── 4c. Training helpers ──────────────────────────────────────────────────────
def to_dev(tensors, mask, labels):
    return [t.to(DEVICE) for t in tensors], mask.to(DEVICE), labels.to(DEVICE)

def recon_loss(recons, tensors, mask):
    total, count = 0.0, 0
    for i, (r, x) in enumerate(zip(recons, tensors)):
        m = mask[:, i]
        if m.sum() == 0: continue
        total += F.mse_loss(r[m==1], x[m==1])
        count += 1
    return total / max(count, 1)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total, all_pred, all_true = 0, 0, [], []
    for tensors, mask, labels in loader:
        tensors, mask, labels = to_dev(tensors, mask, labels)
        logits, _, _ = model(tensors, mask)
        preds = logits.argmax(1)
        correct += (preds==labels).sum().item()
        total   += labels.size(0)
        all_pred.extend(preds.cpu().numpy())
        all_true.extend(labels.cpu().numpy())
    return correct/total, np.array(all_pred), np.array(all_true)

def save_ckpt(model, opt, epoch, val_acc, name):
    torch.save({'epoch':epoch,'val_acc':val_acc,
                'model':model.state_dict(),'opt':opt.state_dict()},
               MDL_DIR/name)
    print(f'  💾 Saved {name}  (val_acc={val_acc:.4f})')

print('✅ Helpers ready')

✅ Helpers ready


In [ ]:
# ── 4d. Phase 1: Autoencoder pretraining ─────────────────────────────────────
print('=== Phase 1: Autoencoder Pretraining ===')
opt_pre = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched   = optim.lr_scheduler.CosineAnnealingLR(opt_pre, T_max=PRETRAIN_EPOCHS)
pre_losses = []

for epoch in range(1, PRETRAIN_EPOCHS+1):
    model.train()
    total = 0.0
    for tensors, mask, labels in train_loader:
        tensors, mask, labels = to_dev(tensors, mask, labels)
        opt_pre.zero_grad()
        _, recons, _ = model(tensors, mask)
        loss = recon_loss(recons, tensors, mask)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt_pre.step()
        total += loss.item()
    avg = total / len(train_loader)
    pre_losses.append(avg)
    sched.step()
    if epoch % 5 == 0 or epoch == 1:
        print(f'  Epoch {epoch:3d}/{PRETRAIN_EPOCHS} | recon_loss={avg:.6f}')

save_ckpt(model, opt_pre, PRETRAIN_EPOCHS, 0, 'pretrained.pt')
plt.figure(figsize=(7,3))
plt.plot(pre_losses); plt.title('Pretraining Loss')
plt.xlabel('Epoch'); plt.ylabel('MSE')
plt.savefig(RES_DIR/'pretrain_loss.png', dpi=120)
plt.show()
print('✅ Pretraining complete')

=== Phase 1: Autoencoder Pretraining ===


KeyboardInterrupt: 

In [ ]:
# ── 4e. Phase 2: Joint fine-tuning ────────────────────────────────────────────
print('=== Phase 2: Joint Fine-tuning ===')
opt  = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
sch  = optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=20, T_mult=2)
ce   = nn.CrossEntropyLoss(label_smoothing=0.05)
best_val_acc = 0.0
history = {'train_loss':[], 'train_acc':[], 'val_acc':[]}

for epoch in range(1, FINETUNE_EPOCHS+1):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for tensors, mask, labels in train_loader:
        tensors, mask, labels = to_dev(tensors, mask, labels)
        opt.zero_grad()
        logits, recons, _ = model(tensors, mask)
        loss = 0.7*ce(logits,labels) + 0.3*recon_loss(recons,tensors,mask)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total_loss += loss.item()
        correct    += (logits.argmax(1)==labels).sum().item()
        total      += labels.size(0)
    train_acc = correct/total
    val_acc, _, _ = evaluate(model, val_loader)
    sch.step()
    history['train_loss'].append(total_loss/len(train_loader))
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_ckpt(model, opt, epoch, val_acc, 'best_model.pt')
    if epoch % 10 == 0:
        save_ckpt(model, opt, epoch, val_acc, f'checkpoint_epoch_{epoch}.pt')
    if epoch % 5 == 0 or epoch == 1:
        print(f'  Epoch {epoch:3d}/{FINETUNE_EPOCHS} | '
              f'loss={history["train_loss"][-1]:.4f} | '
              f'train={train_acc:.3f} | val={val_acc:.3f}')

fig, (ax1,ax2) = plt.subplots(1,2,figsize=(12,4))
ax1.plot(history['train_loss']); ax1.set_title('Loss')
ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'],   label='Val'); ax2.legend(); ax2.set_title('Accuracy')
plt.tight_layout()
plt.savefig(RES_DIR/'training_curves.png', dpi=120)
plt.show()
print(f'\n✅ Training complete — best val acc: {best_val_acc:.4f}')

=== Phase 2: Joint Fine-tuning ===
  💾 Saved best_model.pt  (val_acc=0.9463)
  Epoch   1/80 | loss=0.9033 | train=0.810 | val=0.946
  💾 Saved best_model.pt  (val_acc=0.9575)
  💾 Saved best_model.pt  (val_acc=0.9635)
  💾 Saved best_model.pt  (val_acc=0.9776)
  Epoch   5/80 | loss=0.3688 | train=0.985 | val=0.978
  💾 Saved best_model.pt  (val_acc=0.9799)
  💾 Saved best_model.pt  (val_acc=0.9806)
  💾 Saved checkpoint_epoch_10.pt  (val_acc=0.9799)
  Epoch  10/80 | loss=0.3421 | train=0.999 | val=0.980
  💾 Saved best_model.pt  (val_acc=0.9814)
  Epoch  15/80 | loss=0.3379 | train=1.000 | val=0.980
  💾 Saved checkpoint_epoch_20.pt  (val_acc=0.9791)
  Epoch  20/80 | loss=0.3368 | train=1.000 | val=0.979
  Epoch  25/80 | loss=0.3369 | train=1.000 | val=0.978
  💾 Saved checkpoint_epoch_30.pt  (val_acc=0.9806)
  Epoch  30/80 | loss=0.3328 | train=1.000 | val=0.981
  Epoch  35/80 | loss=0.3306 | train=1.000 | val=0.980
  💾 Saved checkpoint_epoch_40.pt  (val_acc=0.9784)
  Epoch  40/80 | loss=0.329

---
## 📊 Section 5 — Evaluation & Ablation

In [ ]:
# ── 5a. Load best model + test evaluation ─────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from itertools import combinations

ckpt = torch.load(MDL_DIR/'best_model.pt', map_location=DEVICE)
model.load_state_dict(ckpt['model'])
print(f'Loaded best model from epoch {ckpt["epoch"]} (val_acc={ckpt["val_acc"]:.4f})')

test_acc, y_pred, y_true = evaluate(model, test_loader)
print(f'\n🎯 Test Accuracy: {test_acc:.4f}')
print(f'   Macro F1:      {f1_score(y_true,y_pred,average="macro"):.4f}')
print(f'   Weighted F1:   {f1_score(y_true,y_pred,average="weighted"):.4f}')
print()
print(classification_report(y_true, y_pred, target_names=le.classes_, digits=3))

Loaded best model from epoch 70 (val_acc=0.9821)

🎯 Test Accuracy: 0.9797
   Macro F1:      0.9681
   Weighted F1:   0.9795

              precision    recall  f1-score   support

    TCGA-ACC      1.000     0.833     0.909        12
   TCGA-BLCA      1.000     0.951     0.975        61
   TCGA-BRCA      0.994     0.994     0.994       164
   TCGA-CESC      0.957     0.978     0.968        46
   TCGA-CHOL      1.000     0.800     0.889         5
   TCGA-COAD      0.904     0.957     0.930        69
   TCGA-DLBC      0.875     1.000     0.933         7
   TCGA-ESCA      1.000     1.000     1.000        28
    TCGA-GBM      1.000     1.000     1.000        44
   TCGA-HNSC      1.000     0.987     0.994        78
   TCGA-KICH      0.909     1.000     0.952        10
   TCGA-KIRC      0.988     0.988     0.988        80
   TCGA-KIRP      0.976     0.953     0.965        43
   TCGA-LAML      1.000     1.000     1.000        23
    TCGA-LGG      1.000     0.987     0.993        77
   TCGA-LI

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix

# Get per-class breakdown
cm = confusion_matrix(y_true, y_pred)

results = []
for i, cancer in enumerate(le.classes_):
    total       = cm[i].sum()          # total samples of this type
    correct     = cm[i][i]             # correctly classified
    misclassified = total - correct    # wrongly classified
    recall      = correct / total      # % correctly identified

    # What did misclassified samples get predicted as?
    wrong_preds = [(le.classes_[j], cm[i][j])
                   for j in range(len(le.classes_))
                   if j != i and cm[i][j] > 0]
    wrong_preds = sorted(wrong_preds, key=lambda x: -x[1])
    top_confusion = wrong_preds[0][0] if wrong_preds else 'none'

    results.append({
        'Cancer':          cancer.replace('TCGA-',''),
        'Total':           total,
        'Correct':         correct,
        'Misclassified':   misclassified,
        'Recall':          f'{recall:.1%}',
        'Most confused with': top_confusion.replace('TCGA-',''),
    })

df = pd.DataFrame(results).sort_values('Misclassified', ascending=False)
print(df.to_string(index=False))


NameError: name 'y_true' is not defined

In [ ]:
# ── 5b. Confusion matrix ──────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
short = [c.replace('TCGA-','') for c in le.classes_]

fig, ax = plt.subplots(figsize=(17,14))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=short, yticklabels=short,
            vmin=0, vmax=1, linewidths=0.3, ax=ax, annot_kws={'size':7})
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix — Test Accuracy={test_acc:.3f}', fontsize=13)
plt.tight_layout()
plt.savefig(RES_DIR/'confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# ── 5c. UMAP of latent space ──────────────────────────────────────────────────
import umap

model.eval()
all_z, all_y = [], []
with torch.no_grad():
    for tensors, mask, labels in tqdm(test_loader, desc='Extracting latents'):
        tensors, mask, labels = to_dev(tensors, mask, labels)
        fused, _, _ = model.encode_all(tensors, mask)
        all_z.append(fused.cpu().numpy())
        all_y.extend(labels.cpu().numpy())

Z    = np.vstack(all_z)
y_z  = np.array(all_y)
Z_2d = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.1, random_state=42).fit_transform(Z)

fig, ax = plt.subplots(figsize=(14,11))
cmap = plt.cm.get_cmap('tab20', NUM_CLASSES)
for i, ct in enumerate(le.classes_):
    m = y_z == i
    ax.scatter(Z_2d[m,0], Z_2d[m,1], s=8, alpha=0.7, color=cmap(i), label=ct.replace('TCGA-',''))
ax.legend(loc='upper right', ncol=2, fontsize=7, markerscale=2)
ax.set_title('UMAP of Fused Latent Space', fontsize=14)
plt.tight_layout()
plt.savefig(RES_DIR/'umap_latent.png', dpi=150)
plt.show()

Extracting latents:   0%|          | 0/25 [00:00<?, ?it/s]

In [ ]:
# ── 5d. Modality ablation ─────────────────────────────────────────────────────
@torch.no_grad()
def predict_ablated(model, loader, keep_only=None):
    model.eval()
    correct, total = 0, 0
    for tensors, mask, labels in loader:
        tensors, mask, labels = to_dev(tensors, mask, labels)
        tensors = list(tensors)
        mask    = mask.clone()
        if keep_only is not None:
            for i in range(len(tensors)):
                if i not in keep_only:
                    tensors[i] = torch.zeros_like(tensors[i])
                    mask[:,i]  = 0.0
        logits, _, _ = model(tensors, mask)
        correct += (logits.argmax(1)==labels).sum().item()
        total   += labels.size(0)
    return correct/total

print('Running ablation study...')
ablation = {'All (baseline)': test_acc}
for i, name in enumerate(MODALITY_NAMES):
    keep = [j for j in range(len(MODALITY_NAMES)) if j != i]
    acc  = predict_ablated(model, test_loader, keep_only=keep)
    ablation[f'Remove {name}'] = acc
    print(f'  Remove {name:6s}: {acc:.4f}  (drop={test_acc-acc:+.4f})')

print('\nSingle modality:')
single = {}
for i, name in enumerate(MODALITY_NAMES):
    acc = predict_ablated(model, test_loader, keep_only=[i])
    single[name] = acc
    print(f'  {name:6s} alone: {acc:.4f}')

Running ablation study...
  Remove rna   : 0.9113  (drop=+0.0684)
  Remove meth  : 0.9297  (drop=+0.0501)
  Remove cnv   : 0.9544  (drop=+0.0253)
  Remove mirna : 0.9702  (drop=+0.0095)
  Remove mut   : 0.9797  (drop=+0.0000)
  Remove clin  : 0.9791  (drop=+0.0006)

Single modality:
  rna    alone: 0.7250
  meth   alone: 0.6147
  cnv    alone: 0.1039
  mirna  alone: 0.2484
  mut    alone: 0.1039
  clin   alone: 0.1039


In [ ]:
# ── 5e. Pair and triple combinations ─────────────────────────────────────────
print('Testing 2-modality pairs...')
pairs = {}
for i,j in combinations(range(len(MODALITY_NAMES)), 2):
    acc = predict_ablated(model, test_loader, keep_only=[i,j])
    pairs[f'{MODALITY_NAMES[i]}+{MODALITY_NAMES[j]}'] = acc

print('Testing 3-modality triples...')
triples = {}
for combo in combinations(range(len(MODALITY_NAMES)), 3):
    acc = predict_ablated(model, test_loader, keep_only=list(combo))
    triples['+'.join(MODALITY_NAMES[i] for i in combo)] = acc

best_pair   = max(pairs.items(),   key=lambda x: x[1])
best_triple = max(triples.items(), key=lambda x: x[1])

# Plot
fig, axes = plt.subplots(1,3,figsize=(18,5))
single_s = pd.Series(single).sort_values(ascending=False)
axes[0].bar(single_s.index, single_s.values, color='steelblue')
axes[0].axhline(test_acc, color='red', linestyle='--', label=f'All 6 ({test_acc:.3f})')
axes[0].set_title('Single Modality'); axes[0].legend(); axes[0].set_ylim(0,1)
top_pairs = pd.Series(pairs).sort_values(ascending=False).head(15)
axes[1].barh(top_pairs.index[::-1], top_pairs.values[::-1], color='darkorange')
axes[1].axvline(test_acc, color='red', linestyle='--')
axes[1].set_title('Best 2-Modality Pairs')
top_tri = pd.Series(triples).sort_values(ascending=False).head(10)
axes[2].barh(top_tri.index[::-1], top_tri.values[::-1], color='seagreen')
axes[2].axvline(test_acc, color='red', linestyle='--')
axes[2].set_title('Best 3-Modality Triples')
plt.suptitle('Accuracy vs Modality Combinations', fontsize=14)
plt.tight_layout()
plt.savefig(RES_DIR/'modality_combinations.png', dpi=150, bbox_inches='tight')
plt.show()

# Save summary
summary = {
    'test_accuracy': float(test_acc),
    'macro_f1': float(f1_score(y_true,y_pred,average='macro')),
    'single_modality': {k:float(v) for k,v in single.items()},
    'ablation': {k:float(v) for k,v in ablation.items()},
    'best_pair': best_pair[0], 'best_pair_acc': float(best_pair[1]),
    'best_triple': best_triple[0], 'best_triple_acc': float(best_triple[1]),
}
with open(RES_DIR/'summary.json','w') as f:
    json.dump(summary, f, indent=2)

print('\n' + '='*55)
print('         RESULTS SUMMARY')
print('='*55)
print(f'  Test Accuracy (all 6):  {test_acc:.4f}')
print(f'  Best single modality:   {max(single.items(), key=lambda x:x[1])}')
print(f'  Best pair:              {best_pair}')
print(f'  Best triple:            {best_triple}')
print(f'\n  Importance ranking (drop when removed):')
drops = {k.replace("Remove ",""): test_acc-v for k,v in ablation.items() if "All" not in k}
for k,v in sorted(drops.items(), key=lambda x:-x[1]):
    print(f'    {k:8s}  {v:+.4f}')
print('='*55)
print(f'\n✅ All results saved to Drive: {RES_DIR}')

Testing 2-modality pairs...
Testing 3-modality triples...

         RESULTS SUMMARY
  Test Accuracy (all 6):  0.9797
  Best single modality:   ('rna', 0.7249683143219265)
  Best pair:              ('rna+meth', 0.9328263624841572)
  Best triple:            ('rna+meth+cnv', 0.9689480354879595)

  Importance ranking (drop when removed):
    rna       +0.0684
    meth      +0.0501
    cnv       +0.0253
    mirna     +0.0095
    clin      +0.0006
    mut       +0.0000

✅ All results saved to Drive: /content/drive/MyDrive/tcga_pancancer/results


In [ ]:
print('Testing 4-modality combinations...')
quads = {}
for combo in combinations(range(len(MODALITY_NAMES)), 4):
    acc = predict_ablated(model, test_loader, keep_only=list(combo))
    quads['+'.join(MODALITY_NAMES[i] for i in combo)] = acc

top_quads = pd.Series(quads).sort_values(ascending=False)
print('Top 5 four-modality combinations:')
print(top_quads.head(5).to_string())

# Add to the combinations plot
top_q = top_quads.head(10)
plt.figure(figsize=(8,5))
plt.barh(top_q.index[::-1], top_q.values[::-1], color='purple')
plt.axvline(test_acc, color='red', linestyle='--', label=f'All 6 ({test_acc:.3f})')
plt.title('Best 4-Modality Combinations')
plt.xlabel('Test Accuracy')
plt.legend()
plt.tight_layout()
plt.savefig(RES_DIR/'quad_combinations.png', dpi=150)
plt.show()

Testing 4-modality combinations...
Top 5 four-modality combinations:
rna+meth+cnv+mirna     0.979087
rna+meth+cnv+clin      0.970215
rna+meth+cnv+mut       0.968948
rna+meth+mirna+clin    0.954373
rna+meth+mirna+mut     0.951838


In [ ]:
# Get case IDs of misclassified ACC samples
import torch

ckpt = torch.load(MDL_DIR/'best_model.pt', map_location=DEVICE)
model.load_state_dict(ckpt['model'])
print(f'Loaded best model from epoch {ckpt["epoch"]} (val_acc={ckpt["val_acc"]:.4f})')
acc_idx = list(le.classes_).index('TCGA-ACC')

misclassified_acc = []
for i, (true, pred) in enumerate(zip(y_true, y_pred)):
    if true == acc_idx and pred != acc_idx:
        case_id = case_ids[idx_test[i]]
        predicted_as = le.classes_[pred]
        misclassified_acc.append({'case_id': case_id, 'predicted_as': predicted_as})

print(f'Misclassified ACC samples: {len(misclassified_acc)}')
for s in misclassified_acc:
    print(f"  {s['case_id']}  →  predicted as {s['predicted_as']}")

Loaded best model from epoch 70 (val_acc=0.9821)
Misclassified ACC samples: 2
  08e0d412-d4d8-4d13-b792-a4dd0bd9ec2b  →  predicted as TCGA-SARC
  d63f2f1e-1970-445f-907b-23cf4d9efd83  →  predicted as TCGA-LUAD


In [ ]:
acc_idx = list(le.classes_).index('TCGA-ACC')

print(f'Misclassified ACC samples:')
for i, (true, pred) in enumerate(zip(y_true, y_pred)):
    if true == acc_idx and pred != acc_idx:
        case_id      = case_ids[idx_test[i]]
        predicted_as = le.classes_[pred]
        print(f'  {case_id}  →  {predicted_as}')

Misclassified ACC samples:
  08e0d412-d4d8-4d13-b792-a4dd0bd9ec2b  →  TCGA-SARC
  d63f2f1e-1970-445f-907b-23cf4d9efd83  →  TCGA-LUAD


In [ ]:
import requests

acc_idx = list(le.classes_).index('TCGA-ACC')

for i, (true, pred) in enumerate(zip(y_true, y_pred)):
    if true == acc_idx and pred != acc_idx:
        case_id = case_ids[idx_test[i]]
        # Look up TCGA barcode from GDC
        r = requests.get(f'https://api.gdc.cancer.gov/cases/{case_id}',
                        params={'fields': 'submitter_id'})
        barcode = r.json()['data']['submitter_id']
        print(f'  UUID:    {case_id}')
        print(f'  Barcode: {barcode}')
        print(f'  Predicted as: {le.classes_[pred]}')
        print()

  UUID:    08e0d412-d4d8-4d13-b792-a4dd0bd9ec2b
  Barcode: TCGA-OR-A5J8
  Predicted as: TCGA-SARC

  UUID:    d63f2f1e-1970-445f-907b-23cf4d9efd83
  Barcode: TCGA-OR-A5JB
  Predicted as: TCGA-LUAD



In [ ]:
import requests

TCGA_NAMES = {
    'TCGA-ACC':  'Adrenocortical Carcinoma',
    'TCGA-BLCA': 'Bladder Urothelial Carcinoma',
    'TCGA-BRCA': 'Breast Invasive Carcinoma',
    'TCGA-CESC': 'Cervical Squamous Cell Carcinoma',
    'TCGA-CHOL': 'Cholangiocarcinoma',
    'TCGA-COAD': 'Colon Adenocarcinoma',
    'TCGA-DLBC': 'Diffuse Large B-cell Lymphoma',
    'TCGA-ESCA': 'Esophageal Carcinoma',
    'TCGA-GBM':  'Glioblastoma Multiforme',
    'TCGA-HNSC': 'Head and Neck Squamous Cell Carcinoma',
    'TCGA-KICH': 'Kidney Chromophobe',
    'TCGA-KIRC': 'Kidney Renal Clear Cell Carcinoma',
    'TCGA-KIRP': 'Kidney Renal Papillary Cell Carcinoma',
    'TCGA-LAML': 'Acute Myeloid Leukemia',
    'TCGA-LGG':  'Brain Lower Grade Glioma',
    'TCGA-LIHC': 'Liver Hepatocellular Carcinoma',
    'TCGA-LUAD': 'Lung Adenocarcinoma',
    'TCGA-LUSC': 'Lung Squamous Cell Carcinoma',
    'TCGA-MESO': 'Mesothelioma',
    'TCGA-OV':   'Ovarian Serous Cystadenocarcinoma',
    'TCGA-PAAD': 'Pancreatic Adenocarcinoma',
    'TCGA-PCPG': 'Pheochromocytoma and Paraganglioma',
    'TCGA-PRAD': 'Prostate Adenocarcinoma',
    'TCGA-READ': 'Rectum Adenocarcinoma',
    'TCGA-SARC': 'Sarcoma',
    'TCGA-SKCM': 'Skin Cutaneous Melanoma',
    'TCGA-STAD': 'Stomach Adenocarcinoma',
    'TCGA-TGCT': 'Testicular Germ Cell Tumors',
    'TCGA-THCA': 'Thyroid Carcinoma',
    'TCGA-THYM': 'Thymoma',
    'TCGA-UCEC': 'Uterine Corpus Endometrial Carcinoma',
    'TCGA-UCS':  'Uterine Carcinosarcoma',
    'TCGA-UVM':  'Uveal Melanoma',
}

chol_idx = list(le.classes_).index('TCGA-CHOL')

print(f'True label: TCGA-CHOL — Cholangiocarcinoma')
print('='*60)

misclassified_chol = []
for i, (true, pred) in enumerate(zip(y_true, y_pred)):
    if true == chol_idx and pred != chol_idx:
        case_id      = case_ids[idx_test[i]]
        predicted_as = le.classes_[pred]
        misclassified_chol.append({
            'case_id':      case_id,
            'predicted_as': predicted_as,
        })

print(f'Total misclassified: {len(misclassified_chol)}')
print()

for s in misclassified_chol:
    try:
        r       = requests.get(f'https://api.gdc.cancer.gov/cases/{s["case_id"]}',
                               params={'fields': 'submitter_id'}, timeout=10)
        barcode = r.json()['data']['submitter_id']
    except:
        barcode = 'lookup failed'

    print(f'  UUID:         {s["case_id"]}')
    print(f'  TCGA Barcode: {barcode}')
    print(f'  Predicted as: {s["predicted_as"]} — {TCGA_NAMES[s["predicted_as"]]}')
    print()

True label: TCGA-CHOL — Cholangiocarcinoma
Total misclassified: 1

  UUID:         bc15cec4-1f82-47fe-ae69-3853194fab7e
  TCGA Barcode: TCGA-W5-AA2H
  Predicted as: TCGA-THYM — Thymoma



In [ ]:
import requests

chol_idx = list(le.classes_).index('TCGA-CHOL')

print(f'True label: TCGA-CHOL — Cholangiocarcinoma')
print(f'{"="*60}')

misclassified_chol = []
for i, (true, pred) in enumerate(zip(y_true, y_pred)):
    if true == chol_idx and pred != chol_idx:
        case_id      = case_ids[idx_test[i]]
        predicted_as = le.classes_[pred]
        misclassified_chol.append({
            'case_id':      case_id,
            'predicted_as': predicted_as,
        })

print(f'Total misclassified: {len(misclassified_chol)}')
print()

for s in misclassified_chol:
    # Look up TCGA barcode
    try:
        r       = requests.get(f'https://api.gdc.cancer.gov/cases/{s["case_id"]}',
                               params={'fields': 'submitter_id'}, timeout=10)
        barcode = r.json()['data']['submitter_id']
    except:
        barcode = 'lookup failed'

    print(f'  UUID:         {s["case_id"]}')
    print(f'  TCGA Barcode: {barcode}')
    print(f'  Predicted as: {s["predicted_as"]} — {TCGA_NAMES[s["predicted_as"]]}')
    print()

True label: TCGA-CHOL — Cholangiocarcinoma
Total misclassified: 1

  UUID:         bc15cec4-1f82-47fe-ae69-3853194fab7e
  TCGA Barcode: TCGA-W5-AA2H
  Predicted as: TCGA-THYM — Thymoma



In [ ]:
import requests

TCGA_NAMES = {
    'TCGA-ACC':  'Adrenocortical Carcinoma',
    'TCGA-BLCA': 'Bladder Urothelial Carcinoma',
    'TCGA-BRCA': 'Breast Invasive Carcinoma',
    'TCGA-CESC': 'Cervical Squamous Cell Carcinoma',
    'TCGA-CHOL': 'Cholangiocarcinoma',
    'TCGA-COAD': 'Colon Adenocarcinoma',
    'TCGA-DLBC': 'Diffuse Large B-cell Lymphoma',
    'TCGA-ESCA': 'Esophageal Carcinoma',
    'TCGA-GBM':  'Glioblastoma Multiforme',
    'TCGA-HNSC': 'Head and Neck Squamous Cell Carcinoma',
    'TCGA-KICH': 'Kidney Chromophobe',
    'TCGA-KIRC': 'Kidney Renal Clear Cell Carcinoma',
    'TCGA-KIRP': 'Kidney Renal Papillary Cell Carcinoma',
    'TCGA-LAML': 'Acute Myeloid Leukemia',
    'TCGA-LGG':  'Brain Lower Grade Glioma',
    'TCGA-LIHC': 'Liver Hepatocellular Carcinoma',
    'TCGA-LUAD': 'Lung Adenocarcinoma',
    'TCGA-LUSC': 'Lung Squamous Cell Carcinoma',
    'TCGA-MESO': 'Mesothelioma',
    'TCGA-OV':   'Ovarian Serous Cystadenocarcinoma',
    'TCGA-PAAD': 'Pancreatic Adenocarcinoma',
    'TCGA-PCPG': 'Pheochromocytoma and Paraganglioma',
    'TCGA-PRAD': 'Prostate Adenocarcinoma',
    'TCGA-READ': 'Rectum Adenocarcinoma',
    'TCGA-SARC': 'Sarcoma',
    'TCGA-SKCM': 'Skin Cutaneous Melanoma',
    'TCGA-STAD': 'Stomach Adenocarcinoma',
    'TCGA-TGCT': 'Testicular Germ Cell Tumors',
    'TCGA-THCA': 'Thyroid Carcinoma',
    'TCGA-THYM': 'Thymoma',
    'TCGA-UCEC': 'Uterine Corpus Endometrial Carcinoma',
    'TCGA-UCS':  'Uterine Carcinosarcoma',
    'TCGA-UVM':  'Uveal Melanoma',
}

# Collect all misclassified samples
print('Collecting misclassified samples...')
misclassified = []
for i, (true, pred) in enumerate(zip(y_true, y_pred)):
    if true != pred:
        case_id    = case_ids[idx_test[i]]
        true_label = le.classes_[true]
        pred_label = le.classes_[pred]
        misclassified.append({
            'case_id':      case_id,
            'true_label':   true_label,
            'pred_label':   pred_label,
        })

print(f'Total misclassified: {len(misclassified)}')
print(f'Total test samples:  {len(y_true)}')
print(f'Error rate:          {len(misclassified)/len(y_true):.2%}')
print()

# Look up TCGA barcodes for all misclassified samples
print('Looking up TCGA barcodes (this may take a few minutes)...')
for i, s in enumerate(misclassified):
    try:
        r       = requests.get(f'https://api.gdc.cancer.gov/cases/{s["case_id"]}',
                               params={'fields': 'submitter_id'}, timeout=10)
        s['barcode'] = r.json()['data']['submitter_id']
    except:
        s['barcode'] = 'lookup failed'
    if (i+1) % 10 == 0:
        print(f'  {i+1}/{len(misclassified)} done...')

# Print grouped by true cancer type
print()
print('='*70)
print('ALL MISCLASSIFIED SAMPLES')
print('='*70)

current_type = None
for s in sorted(misclassified, key=lambda x: x['true_label']):
    if s['true_label'] != current_type:
        current_type = s['true_label']
        count = sum(1 for x in misclassified if x['true_label'] == current_type)
        print(f"\n{current_type} — {TCGA_NAMES[current_type]} ({count} misclassified)")
        print('-'*50)
    print(f"  Barcode:      {s['barcode']}")
    print(f"  UUID:         {s['case_id']}")
    print(f"  Predicted as: {s['pred_label']} — {TCGA_NAMES[s['pred_label']]}")
    print()

# Save to CSV on Drive
import pandas as pd
df = pd.DataFrame(misclassified)
df['true_name'] = df['true_label'].map(TCGA_NAMES)
df['pred_name'] = df['pred_label'].map(TCGA_NAMES)
df.to_csv('/content/drive/MyDrive/tcga_pancancer/results/misclassified_samples.csv', index=False)
print('✅ Saved to results/misclassified_samples.csv')

Total misclassified: 32
Total test samples:  1578
Error rate:          2.03%

Looking up TCGA barcodes (this may take a few minutes)...
  10/32 done...
  20/32 done...
  30/32 done...

ALL MISCLASSIFIED SAMPLES

TCGA-ACC — Adrenocortical Carcinoma (2 misclassified)
--------------------------------------------------
  Barcode:      TCGA-OR-A5J8
  UUID:         08e0d412-d4d8-4d13-b792-a4dd0bd9ec2b
  Predicted as: TCGA-SARC — Sarcoma

  Barcode:      TCGA-OR-A5JB
  UUID:         d63f2f1e-1970-445f-907b-23cf4d9efd83
  Predicted as: TCGA-LUAD — Lung Adenocarcinoma


TCGA-BLCA — Bladder Urothelial Carcinoma (3 misclassified)
--------------------------------------------------
  Barcode:      TCGA-FD-A3B4
  UUID:         899f8d52-4c3a-4371-90eb-1f106001eddc
  Predicted as: TCGA-CESC — Cervical Squamous Cell Carcinoma

  Barcode:      TCGA-GD-A3OS
  UUID:         2a41b7d4-132f-4fb6-87b5-487c445379c7
  Predicted as: TCGA-MESO — Mesothelioma

  Barcode:      TCGA-DK-AA6T
  UUID:         f9b55748-

In [ ]:
from pathlib import Path

results_dir = Path('/content/drive/MyDrive/tcga_pancancer/results')
for f in sorted(results_dir.glob('*.png')):
    print(f.name)

confusion_matrix.png
modality_combinations.png
pretrain_loss.png
quad_combinations.png
training_curves.png
umap_latent.png


In [ ]:
import requests
from tqdm import tqdm

# Get UUIDs directly from the mod_maps (already loaded in recovery)
rna_uuids = list(mod_maps[0].keys())  # mod_maps[0] = RNA index → row mapping
print(f"Total UUIDs to map: {len(rna_uuids)}")

GDC_CASES_URL = "https://api.gdc.cancer.gov/cases"
BATCH_SIZE = 500
results = []

for i in tqdm(range(0, len(rna_uuids), BATCH_SIZE)):
    batch = rna_uuids[i:i+BATCH_SIZE]
    payload = {
        "filters": {"op": "in", "content": {"field": "case_id", "value": batch}},
        "fields": "case_id,submitter_id",
        "format": "JSON",
        "size": str(BATCH_SIZE)
    }
    response = requests.post(GDC_CASES_URL, json=payload)
    if response.status_code == 200:
        for hit in response.json().get("data", {}).get("hits", []):
            results.append({"case_id": hit["case_id"], "barcode": hit["submitter_id"]})
    else:
        print(f"Batch {i//BATCH_SIZE} failed: {response.status_code}")

uuid_map = pd.DataFrame(results)
print(f"\nMapped {len(uuid_map)} / {len(rna_uuids)} UUIDs")
uuid_map.to_csv(PROC_DIR / 'subtypes' / 'uuid_to_barcode.csv', index=False)
print("Saved. Sample:")
print(uuid_map.head())

Total UUIDs to map: 10517


100%|██████████| 22/22 [00:35<00:00,  1.59s/it]


Mapped 10517 / 10517 UUIDs
Saved. Sample:
                                case_id       barcode
0  0f64edec-0f1f-4025-8a53-75f9534f7828  TCGA-BH-A0H9
1  5258294f-ef5b-4091-9183-6bcc545d54b2  TCGA-BH-A209
2  2476dda9-da2a-4242-9642-b92b8b9b1c79  TCGA-B6-A0IH
3  359f12f9-5c41-48a4-85bc-fd7e307bf7d8  TCGA-EW-A1P8
4  bb026999-0aab-4232-96d7-2c93b6c9691d  TCGA-A2-A3KD


In [ ]:
uuid_map = pd.read_csv(PROC_DIR / 'subtypes' / 'uuid_to_barcode.csv')
subtypes = pd.read_csv(PROC_DIR / 'subtypes' / 'molecular_subtypes.csv')

# Trim sampleID to 12-char patient barcode for matching
subtypes['patient_barcode'] = subtypes['sampleID'].str[:12]

# Extract cancer type from Subtype_Selected prefix (e.g. "ACC.CIMP-high" → "TCGA-ACC")
subtypes['cancer_type'] = 'TCGA-' + subtypes['Subtype_Selected'].str.split('.').str[0]

merged = uuid_map.merge(
    subtypes[['patient_barcode', 'cancer_type', 'Subtype_Selected']],
    left_on='barcode',
    right_on='patient_barcode',
    how='inner'
)

print(f"Samples with subtype labels: {len(merged)} / {len(uuid_map)}")
print(f"\nSubtypes per cancer type:")
print(merged.groupby('cancer_type')['Subtype_Selected'].nunique().sort_values(ascending=False).to_string())
print(f"\nSample counts per cancer type:")
print(merged.groupby('cancer_type').size().sort_values(ascending=False).to_string())

merged.to_csv(PROC_DIR / 'subtypes' / 'rna_with_subtypes.csv', index=False)
print("\nSaved rna_with_subtypes.csv")

Samples with subtype labels: 7280 / 10517

Subtypes per cancer type:
cancer_type
TCGA-PRAD       8
TCGA-GBM_LGG    8
TCGA-AML        7
TCGA-LUAD       6
TCGA-THCA       6
TCGA-GI         6
TCGA-BRCA       5
TCGA-SKCM       5
TCGA-UCEC       5
TCGA-KIRC       5
TCGA-PCPG       5
TCGA-BLCA       4
TCGA-LUSC       4
TCGA-LIHC       4
TCGA-HNSC       4
TCGA-KIRP       4
TCGA-OVCA       4
TCGA-ACC        3
TCGA-KICH       2
TCGA-UCS        2

Sample counts per cancer type:
cancer_type
TCGA-BRCA       1216
TCGA-GI         1008
TCGA-GBM_LGG     804
TCGA-UCEC        536
TCGA-THCA        494
TCGA-KIRC        441
TCGA-OVCA        417
TCGA-PRAD        333
TCGA-SKCM        332
TCGA-HNSC        279
TCGA-LUAD        230
TCGA-LIHC        193
TCGA-PCPG        178
TCGA-LUSC        178
TCGA-KIRP        161
TCGA-AML         150
TCGA-BLCA        129
TCGA-ACC          78
TCGA-KICH         66
TCGA-UCS          57

Saved rna_with_subtypes.csv


In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def per_class_recall(pred, true):
    cm = confusion_matrix(true, pred, labels=list(range(NUM_CLASSES)))
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where(cm.sum(axis=1) > 0,
                        cm.diagonal() / cm.sum(axis=1), 0.0)

baseline_recall = per_class_recall(y_pred, y_true)
recall_drops = {}
for i, name in enumerate(MODALITY_NAMES):
    test_preds = []
    test_true = []
    for tensors, mask, labels in test_loader:
        tensors, mask, labels = to_dev(tensors, mask, labels)
        tensors = list(tensors)
        mask = mask.clone()
        tensors[i] = torch.zeros_like(tensors[i])
        mask[:, i] = 0.0
        with torch.no_grad():
            logits, _, _ = model(tensors, mask)
        test_preds.extend(logits.argmax(1).cpu().numpy())
        test_true.extend(labels.cpu().numpy())
    recall_drops[name] = baseline_recall - per_class_recall(
        np.array(test_preds), np.array(test_true))

short = [c.replace('TCGA-','') for c in le.classes_]
drop_df = pd.DataFrame(recall_drops, index=short)

fig, ax = plt.subplots(figsize=(10, 14))
sns.heatmap(drop_df, annot=True, fmt='.2f', cmap='RdYlGn_r',
            center=0, vmin=-0.1, vmax=0.3, linewidths=0.3, ax=ax,
            annot_kws={'size': 8})
ax.set_title('Recall Drop per Cancer Type when Modality Removed', fontsize=11)
ax.set_xlabel('Modality Removed')
ax.set_ylabel('Cancer Type')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/tcga_pancancer/results/per_cancer_ablation.png', dpi=150)
plt.show()
print('✅ Saved')

✅ Saved


In [ ]:
# ── Subtype Dataset Builder ───────────────────────────────────────────────────
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

merged = pd.read_csv(PROC_DIR / 'subtypes' / 'rna_with_subtypes.csv')

TARGET_CANCERS = ['TCGA-BRCA', 'TCGA-GBM_LGG', 'TCGA-UCEC', 'TCGA-THCA', 'TCGA-KIRC']

subtype_datasets = {}

for cancer in TARGET_CANCERS:
    df = merged[merged['cancer_type'] == cancer].copy()

    # Drop rare subtypes (< 10 samples) — too few to train/eval on
    counts = df['Subtype_Selected'].value_counts()
    valid_subtypes = counts[counts >= 10].index
    df = df[df['Subtype_Selected'].isin(valid_subtypes)].copy()

    # Build label encoder for this cancer
    le_sub = LabelEncoder()
    df['subtype_label'] = le_sub.fit_transform(df['Subtype_Selected'])

    # Map barcode → RNA row index (mod_maps[0] is RNA UUID→row)
    # uuid_map links case_id (UUID) → barcode; merged has barcode
    # We need: barcode → case_id → row index in mod_data[0]
    barcode_to_uuid = dict(zip(uuid_map['barcode'], uuid_map['case_id']))
    df['uuid'] = df['barcode'].map(barcode_to_uuid)
    df = df.dropna(subset=['uuid'])
    df['rna_row'] = df['uuid'].map(mod_maps[0])
    df = df.dropna(subset=['rna_row'])
    df['rna_row'] = df['rna_row'].astype(int)

    subtype_datasets[cancer] = {
        'df': df.reset_index(drop=True),
        'le': le_sub,
        'n_subtypes': len(le_sub.classes_),
        'classes': list(le_sub.classes_),
    }

    print(f"\n{cancer} — {len(df)} samples, {len(le_sub.classes_)} subtypes")
    for cls, count in df['Subtype_Selected'].value_counts().items():
        print(f"  {cls:<35} {count}")

# Save for later cells
with open(PROC_DIR / 'subtypes' / 'subtype_datasets.pkl', 'wb') as f:
    pickle.dump(subtype_datasets, f)
print("\n✅ Saved subtype_datasets.pkl")


TCGA-BRCA — 1216 samples, 5 subtypes
  BRCA.LumA                           581
  BRCA.LumB                           219
  BRCA.Basal                          191
  BRCA.Normal                         143
  BRCA.Her2                           82

TCGA-GBM_LGG — 804 samples, 8 subtypes
  GBM_LGG.G-CIMP-high                 242
  GBM_LGG.Codel                       173
  GBM_LGG.Mesenchymal-like            139
  GBM_LGG.Classic-like                95
  GBM_LGG.NA                          81
  GBM_LGG.LGm6-GBM                    27
  GBM_LGG.PA-like                     26
  GBM_LGG.G-CIMP-low                  21

TCGA-UCEC — 536 samples, 5 subtypes
  UCEC.CN_HIGH                        160
  UCEC.CN_LOW                         144
  UCEC.MSI                            124
  UCEC.POLE                           79
  UCEC.NA                             29

TCGA-THCA — 494 samples, 6 subtypes
  THCA.1                              134
  THCA.4                              106
  THCA.5        

In [ ]:
# ── Drop NA subtypes and rebuild subtype_datasets ────────────────────────────
subtype_datasets_clean = {}

for cancer in TARGET_CANCERS:
    df = merged[merged['cancer_type'] == cancer].copy()

    # Drop NA subtypes
    df = df[~df['Subtype_Selected'].str.endswith('.NA')].copy()

    # Drop rare subtypes < 10 samples
    counts = df['Subtype_Selected'].value_counts()
    valid = counts[counts >= 10].index
    df = df[df['Subtype_Selected'].isin(valid)].copy()

    le_sub = LabelEncoder()
    df['subtype_label'] = le_sub.fit_transform(df['Subtype_Selected'])

    barcode_to_uuid = dict(zip(uuid_map['barcode'], uuid_map['case_id']))
    df['uuid'] = df['barcode'].map(barcode_to_uuid)
    df = df.dropna(subset=['uuid'])
    df['rna_row'] = df['uuid'].map(mod_maps[0])
    df = df.dropna(subset=['rna_row'])
    df['rna_row'] = df['rna_row'].astype(int)

    subtype_datasets_clean[cancer] = {
        'df': df.reset_index(drop=True),
        'le': le_sub,
        'n_subtypes': len(le_sub.classes_),
        'classes': list(le_sub.classes_),
    }

    print(f"\n{cancer} — {len(df)} samples, {len(le_sub.classes_)} subtypes")
    for cls, count in df['Subtype_Selected'].value_counts().items():
        print(f"  {cls:<35} {count}")

# Overwrite and save
subtype_datasets = subtype_datasets_clean
with open(PROC_DIR / 'subtypes' / 'subtype_datasets.pkl', 'wb') as f:
    pickle.dump(subtype_datasets, f)
print("\n✅ Saved cleaned subtype_datasets.pkl")


TCGA-BRCA — 1216 samples, 5 subtypes
  BRCA.LumA                           581
  BRCA.LumB                           219
  BRCA.Basal                          191
  BRCA.Normal                         143
  BRCA.Her2                           82

TCGA-GBM_LGG — 723 samples, 7 subtypes
  GBM_LGG.G-CIMP-high                 242
  GBM_LGG.Codel                       173
  GBM_LGG.Mesenchymal-like            139
  GBM_LGG.Classic-like                95
  GBM_LGG.LGm6-GBM                    27
  GBM_LGG.PA-like                     26
  GBM_LGG.G-CIMP-low                  21

TCGA-UCEC — 507 samples, 4 subtypes
  UCEC.CN_HIGH                        160
  UCEC.CN_LOW                         144
  UCEC.MSI                            124
  UCEC.POLE                           79

TCGA-THCA — 482 samples, 5 subtypes
  THCA.1                              134
  THCA.4                              106
  THCA.5                              93
  THCA.3                              82
  THCA.2        

In [ ]:
# ── Subtype Classifier Head + Training ───────────────────────────────────────
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import torch.nn as nn
import torch

class SubtypeDataset(Dataset):
    def __init__(self, rows, labels, mod_data, mod_maps, case_ids_for_rows):
        # rows: list of rna_row indices into mod_data[0]
        # We'll look up all modalities by matching case_id
        self.rows     = rows
        self.labels   = labels
        self.mod_data = mod_data
        self.mod_maps = mod_maps
        self.cids     = case_ids_for_rows  # case_id (UUID) per sample

    def __len__(self): return len(self.rows)

    def __getitem__(self, i):
        cid = self.cids[i]
        tensors, mask = [], []
        for data, mapping in zip(self.mod_data, self.mod_maps):
            if cid in mapping:
                tensors.append(torch.tensor(data[mapping[cid]], dtype=torch.float32))
                mask.append(1.0)
            else:
                tensors.append(torch.zeros(data.shape[1], dtype=torch.float32))
                mask.append(0.0)
        return tensors, torch.tensor(mask, dtype=torch.float32), torch.tensor(self.labels[i], dtype=torch.long)

class SubtypeClassifier(nn.Module):
    def __init__(self, backbone, n_subtypes, fusion_dim=960, freeze_encoders=True):
        super().__init__()
        self.backbone = backbone
        if freeze_encoders:
            for p in backbone.encoders.parameters():
                p.requires_grad = False
            for p in backbone.gate.parameters():
                p.requires_grad = False
        # Fresh classification head
        self.head = nn.Sequential(
            nn.Linear(fusion_dim, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, 128),        nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, n_subtypes),
        )

    def forward(self, tensors, mask):
        fused, _, _ = self.backbone.encode_all(tensors, mask)
        return self.head(fused.detach())  # detach = frozen backbone grads

def train_subtype_classifier(cancer, subtype_datasets, model, mod_data, mod_maps,
                              epochs=50, lr=1e-3, batch_size=32):
    ds    = subtype_datasets[cancer]
    df    = ds['df']
    n_sub = ds['n_subtypes']

    # Train/val/test split stratified by subtype
    idx   = np.arange(len(df))
    y_all = df['subtype_label'].values
    cids  = df['uuid'].values

    idx_tv, idx_test = train_test_split(idx, test_size=0.15, stratify=y_all, random_state=42)
    idx_train, idx_val = train_test_split(idx_tv, test_size=0.15/0.85,
                                          stratify=y_all[idx_tv], random_state=42)

    def make_loader(split_idx, shuffle=False, weighted=False):
        cids_split   = cids[split_idx]
        labels_split = y_all[split_idx]
        dataset = SubtypeDataset(split_idx, labels_split, mod_data, mod_maps, cids_split)
        sampler = None
        if weighted:
            counts  = np.bincount(labels_split)
            weights = 1.0 / counts[labels_split]
            sampler = WeightedRandomSampler(weights, len(weights))
            shuffle = False
        return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                          sampler=sampler, collate_fn=collate_fn)

    train_loader = make_loader(idx_train, weighted=True)
    val_loader   = make_loader(idx_val)
    test_loader  = make_loader(idx_test)

    clf = SubtypeClassifier(model, n_subtypes=n_sub).to(DEVICE)
    opt = torch.optim.AdamW(clf.head.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    best_val_f1, best_state, history = 0.0, None, []

    for epoch in range(1, epochs + 1):
        clf.train()
        for tensors, mask, labels in train_loader:
            tensors = [t.to(DEVICE) for t in tensors]
            mask, labels = mask.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad()
            logits = clf(tensors, mask)
            nn.CrossEntropyLoss()(logits, labels).backward()
            opt.step()
        sched.step()

        # Validation
        clf.eval()
        preds, trues = [], []
        with torch.no_grad():
            for tensors, mask, labels in val_loader:
                tensors = [t.to(DEVICE) for t in tensors]
                mask = mask.to(DEVICE)
                p = clf(tensors, mask).argmax(1).cpu().numpy()
                preds.extend(p); trues.extend(labels.numpy())
        val_f1 = f1_score(trues, preds, average='macro')
        history.append(val_f1)
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state  = {k: v.clone() for k, v in clf.head.state_dict().items()}
        if epoch % 10 == 0:
            print(f"  [{cancer}] epoch {epoch:3d} | val macro-F1 {val_f1:.4f}")

    # Test evaluation with best head
    clf.head.load_state_dict(best_state)
    clf.eval()
    preds, trues = [], []
    with torch.no_grad():
        for tensors, mask, labels in test_loader:
            tensors = [t.to(DEVICE) for t in tensors]
            mask = mask.to(DEVICE)
            p = clf(tensors, mask).argmax(1).cpu().numpy()
            preds.extend(p); trues.extend(labels.numpy())

    print(f"\n{'='*55}")
    print(f"{cancer} — Test Results (best val F1={best_val_f1:.4f})")
    print(f"{'='*55}")
    print(classification_report(trues, preds,
                                 target_names=ds['classes'], digits=3))

    # Save
    save_path = MDL_DIR / f'subtype_head_{cancer.replace("-","_").replace("/","_")}.pt'
    torch.save({'head': best_state, 'classes': ds['classes'], 'cancer': cancer}, save_path)
    print(f"Saved → {save_path.name}")

    return {'cancer': cancer, 'test_preds': preds, 'test_true': trues,
            'classes': ds['classes'], 'history': history, 'best_val_f1': best_val_f1}

# ── Run all 5 cancers ─────────────────────────────────────────────────────────
all_results = {}
for cancer in TARGET_CANCERS:
    print(f"\nTraining {cancer}...")
    all_results[cancer] = train_subtype_classifier(
        cancer, subtype_datasets, model, mod_data, mod_maps
    )

print("\n✅ All subtype classifiers trained")


Training TCGA-BRCA...
  [TCGA-BRCA] epoch  10 | val macro-F1 0.6812
  [TCGA-BRCA] epoch  20 | val macro-F1 0.6589
  [TCGA-BRCA] epoch  30 | val macro-F1 0.6385
  [TCGA-BRCA] epoch  40 | val macro-F1 0.6478
  [TCGA-BRCA] epoch  50 | val macro-F1 0.6540

TCGA-BRCA — Test Results (best val F1=0.7031)
              precision    recall  f1-score   support

  BRCA.Basal      0.962     0.862     0.909        29
   BRCA.Her2      0.588     0.833     0.690        12
   BRCA.LumA      0.778     0.724     0.750        87
   BRCA.LumB      0.727     0.485     0.582        33
 BRCA.Normal      0.378     0.636     0.475        22

    accuracy                          0.699       183
   macro avg      0.687     0.708     0.681       183
weighted avg      0.737     0.699     0.708       183

Saved → subtype_head_TCGA_BRCA.pt

Training TCGA-GBM_LGG...
  [TCGA-GBM_LGG] epoch  10 | val macro-F1 0.8662
  [TCGA-GBM_LGG] epoch  20 | val macro-F1 0.8836
  [TCGA-GBM_LGG] epoch  30 | val macro-F1 0.8803
  [T

In [ ]:
# ── Required classes for ablation (run after recovery) ───────────────────────
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

class SubtypeDataset(Dataset):
    def __init__(self, rows, labels, mod_data, mod_maps, case_ids_for_rows):
        self.rows     = rows
        self.labels   = labels
        self.mod_data = mod_data
        self.mod_maps = mod_maps
        self.cids     = case_ids_for_rows

    def __len__(self): return len(self.rows)

    def __getitem__(self, i):
        cid = self.cids[i]
        tensors, mask = [], []
        for data, mapping in zip(self.mod_data, self.mod_maps):
            if cid in mapping:
                tensors.append(torch.tensor(data[mapping[cid]], dtype=torch.float32))
                mask.append(1.0)
            else:
                tensors.append(torch.zeros(data.shape[1], dtype=torch.float32))
                mask.append(0.0)
        return tensors, torch.tensor(mask, dtype=torch.float32), torch.tensor(self.labels[i], dtype=torch.long)

def collate_fn(batch):
    tensors_list, masks, labels = zip(*batch)
    return ([torch.stack([s[m] for s in tensors_list]) for m in range(len(tensors_list[0]))],
            torch.stack(masks), torch.stack(labels))

print("✅ SubtypeDataset and collate_fn ready")

✅ SubtypeDataset and collate_fn ready


In [ ]:
# ── Subtype Ablation Study — Retrain per Modality Combination ────────────────
from itertools import combinations
import pickle, time

# Load clean subtype datasets
with open(PROC_DIR / 'subtypes' / 'subtype_datasets.pkl', 'rb') as f:
    subtype_datasets = pickle.load(f)

MODALITY_NAMES = ['rna', 'meth', 'cnv', 'mirna', 'mut', 'clin']
TARGET_CANCERS  = ['TCGA-BRCA', 'TCGA-GBM_LGG', 'TCGA-UCEC', 'TCGA-THCA', 'TCGA-KIRC']

# All 63 non-empty subsets
all_combos = []
for r in range(1, len(MODALITY_NAMES) + 1):
    for combo in combinations(range(len(MODALITY_NAMES)), r):
        all_combos.append(combo)
print(f"Total combinations: {len(all_combos)}")  # should be 63

# ── Mini autoencoder for a given modality subset ──────────────────────────────
class SubsetMultiModalModel(nn.Module):
    def __init__(self, active_indices, mod_dims, latent_dims, n_classes):
        super().__init__()
        self.active = active_indices
        mod_names = ['rna', 'meth', 'cnv', 'mirna', 'mut', 'clin']
        hidden_cfg = {
            'rna':   [2048,1024,512], 'meth':  [2048,1024,512],
            'cnv':   [256,128],       'mirna': [512,256],
            'mut':   [512,256],       'clin':  [64,64],
        }
        self.encoders = nn.ModuleList([
            make_encoder(mod_dims[mod_names[i]],
                         latent_dims[mod_names[i]],
                         hidden_cfg[mod_names[i]])
            for i in active_indices
        ])
        self.gate = nn.Linear(len(active_indices), len(active_indices))
        fusion_dim = sum(latent_dims[mod_names[i]] for i in active_indices)
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, 128),        nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, n_classes),
        )

    def forward(self, tensors, mask):
        # tensors is full 6-modality list; pick active ones
        active_tensors = [tensors[i] for i in self.active]
        active_mask    = mask[:, list(self.active)]
        zs   = [enc(x) for enc, x in zip(self.encoders, active_tensors)]
        gate = torch.sigmoid(self.gate(active_mask))
        fused = torch.cat([z * gate[:, i:i+1] for i, z in enumerate(zs)], dim=-1)
        return self.classifier(fused)


def train_ablation_model(active_indices, cancer, subtype_datasets,
                          mod_data, mod_maps, epochs=60, lr=1e-3, batch_size=32):
    ds    = subtype_datasets[cancer]
    df    = ds['df']
    n_sub = ds['n_subtypes']
    y_all = df['subtype_label'].values
    cids  = df['uuid'].values
    idx   = np.arange(len(df))

    idx_tv, idx_test  = train_test_split(idx, test_size=0.15, stratify=y_all, random_state=42)
    idx_train, idx_val = train_test_split(idx_tv, test_size=0.15/0.85,
                                          stratify=y_all[idx_tv], random_state=42)

    def make_loader(split_idx, weighted=False):
        labels = y_all[split_idx]
        dataset = SubtypeDataset(split_idx, labels, mod_data, mod_maps, cids[split_idx])
        sampler = None
        if weighted:
            counts  = np.bincount(labels)
            weights = 1.0 / counts[labels]
            sampler = WeightedRandomSampler(weights, len(weights))
        return DataLoader(dataset, batch_size=batch_size, shuffle=not weighted,
                          sampler=sampler, collate_fn=collate_fn)

    train_loader = make_loader(idx_train, weighted=True)
    val_loader   = make_loader(idx_val)
    test_loader  = make_loader(idx_test)

    abl_model = SubsetMultiModalModel(active_indices, mod_dims, LATENT_DIMS, n_sub).to(DEVICE)
    opt   = torch.optim.AdamW(abl_model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    best_val_f1, best_state = 0.0, None

    for epoch in range(1, epochs + 1):
        abl_model.train()
        for tensors, mask, labels in train_loader:
            tensors = [t.to(DEVICE) for t in tensors]
            mask, labels = mask.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad()
            logits = abl_model(tensors, mask)
            nn.CrossEntropyLoss()(logits, labels).backward()
            opt.step()
        sched.step()

        abl_model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for tensors, mask, labels in val_loader:
                tensors = [t.to(DEVICE) for t in tensors]
                mask = mask.to(DEVICE)
                p = abl_model(tensors, mask).argmax(1).cpu().numpy()
                preds.extend(p); trues.extend(labels.numpy())
        val_f1 = f1_score(trues, preds, average='macro')
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state  = {k: v.clone() for k, v in abl_model.state_dict().items()}

    # Test eval
    abl_model.load_state_dict(best_state)
    abl_model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for tensors, mask, labels in test_loader:
            tensors = [t.to(DEVICE) for t in tensors]
            mask = mask.to(DEVICE)
            p = abl_model(tensors, mask).argmax(1).cpu().numpy()
            preds.extend(p); trues.extend(labels.numpy())

    return f1_score(trues, preds, average='macro')


# ── Main ablation loop ────────────────────────────────────────────────────────
ablation_results = {}   # {cancer: {combo_tuple: test_macro_f1}}

# Resume support — load existing results if partially done
abl_save_path = PROC_DIR / 'subtypes' / 'ablation_results.pkl'
if abl_save_path.exists():
    with open(abl_save_path, 'rb') as f:
        ablation_results = pickle.load(f)
    print(f"Resuming — loaded existing results")

for cancer in TARGET_CANCERS:
    if cancer not in ablation_results:
        ablation_results[cancer] = {}

    n_done    = len(ablation_results[cancer])
    n_total   = len(all_combos)
    remaining = [c for c in all_combos if c not in ablation_results[cancer]]
    print(f"\n{'='*55}")
    print(f"{cancer} — {n_done}/{n_total} done, {len(remaining)} remaining")
    print(f"{'='*55}")

    for combo in remaining:
        combo_name = '+'.join(MODALITY_NAMES[i] for i in combo)
        t0 = time.time()
        test_f1 = train_ablation_model(combo, cancer, subtype_datasets,
                                        mod_data, mod_maps)
        elapsed = time.time() - t0
        ablation_results[cancer][combo] = test_f1
        print(f"  {combo_name:<45} F1={test_f1:.4f}  ({elapsed:.0f}s)")

        # Save after every model in case of disconnect
        with open(abl_save_path, 'wb') as f:
            pickle.dump(ablation_results, f)

print("\n✅ Ablation complete")

Total combinations: 63
Resuming — loaded existing results

TCGA-BRCA — 63/63 done, 0 remaining

TCGA-GBM_LGG — 63/63 done, 0 remaining

TCGA-UCEC — 63/63 done, 0 remaining

TCGA-THCA — 63/63 done, 0 remaining

TCGA-KIRC — 63/63 done, 0 remaining

✅ Ablation complete


In [ ]:
# ── Ablation Analysis & Figures ───────────────────────────────────────────────
import pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from itertools import combinations

# Load results
abl_save_path = PROC_DIR / 'subtypes' / 'ablation_results.pkl'
with open(abl_save_path, 'rb') as f:
    ablation_results = pickle.load(f)

MODALITY_NAMES = ['rna', 'meth', 'cnv', 'mirna', 'mut', 'clin']
TARGET_CANCERS  = ['TCGA-BRCA', 'TCGA-GBM_LGG', 'TCGA-UCEC', 'TCGA-THCA', 'TCGA-KIRC']
FIG_DIR = RES_DIR / 'ablation_figures'
FIG_DIR.mkdir(exist_ok=True)

CANCER_LABELS = {
    'TCGA-BRCA':    'BRCA',
    'TCGA-GBM_LGG': 'GBM/LGG',
    'TCGA-UCEC':    'UCEC',
    'TCGA-THCA':    'THCA',
    'TCGA-KIRC':    'KIRC',
}
MOD_COLORS = {
    'rna':   '#2E86AB', 'meth':  '#A23B72', 'cnv':   '#F18F01',
    'mirna': '#C73E1D', 'mut':   '#3B1F2B', 'clin':  '#44BBA4',
}

# ── Figure 1: Single-modality bar chart per cancer ────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(16, 4), sharey=False)
fig.suptitle('Single-Modality Subtype Classification (Macro F1)', fontsize=13, fontweight='bold', y=1.02)

for ax, cancer in zip(axes, TARGET_CANCERS):
    single = {MODALITY_NAMES[i]: ablation_results[cancer][(i,)] for i in range(6)}
    mods   = list(single.keys())
    f1s    = list(single.values())
    colors = [MOD_COLORS[m] for m in mods]
    bars   = ax.bar(mods, f1s, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_title(CANCER_LABELS[cancer], fontweight='bold', fontsize=11)
    ax.set_ylim(0, 1.0)
    ax.set_ylabel('Macro F1' if ax == axes[0] else '')
    ax.set_xticklabels(mods, rotation=45, ha='right', fontsize=9)
    ax.axhline(0.2, color='gray', linewidth=0.5, linestyle='--', alpha=0.5)
    for bar, val in zip(bars, f1s):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=7.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig1_single_modality.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved fig1_single_modality.png")


# ── Figure 2: Heatmap — all 63 combos × 5 cancers ────────────────────────────
all_combos = []
for r in range(1, 7):
    for combo in combinations(range(6), r):
        all_combos.append(combo)

combo_labels = ['+'.join(MODALITY_NAMES[i] for i in c) for c in all_combos]
heatmap_data = pd.DataFrame(index=combo_labels, columns=[CANCER_LABELS[c] for c in TARGET_CANCERS])

for cancer in TARGET_CANCERS:
    for combo in all_combos:
        label = '+'.join(MODALITY_NAMES[i] for i in combo)
        heatmap_data.loc[label, CANCER_LABELS[cancer]] = ablation_results[cancer].get(combo, np.nan)

heatmap_data = heatmap_data.astype(float)

fig, ax = plt.subplots(figsize=(7, 22))
sns.heatmap(heatmap_data, ax=ax, cmap='RdYlGn', vmin=0.0, vmax=1.0,
            annot=True, fmt='.2f', annot_kws={'size': 6.5},
            linewidths=0.3, linecolor='#eeeeee',
            cbar_kws={'label': 'Macro F1', 'shrink': 0.3})
ax.set_title('All 63 Modality Combinations — Subtype Macro F1', fontweight='bold', fontsize=12, pad=12)
ax.set_xlabel('')
ax.set_ylabel('Modality Combination', fontsize=9)
ax.tick_params(axis='y', labelsize=7)
ax.tick_params(axis='x', labelsize=10)

# Draw separator lines between modality-count groups
group_sizes = [6, 15, 20, 15, 6, 1]
pos = 0
for gs in group_sizes[:-1]:
    pos += gs
    ax.axhline(pos, color='black', linewidth=1.2)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig2_full_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved fig2_full_heatmap.png")


# ── Figure 3: Best combo per modality count (performance vs complexity) ───────
fig, axes = plt.subplots(1, 5, figsize=(16, 4), sharey=True)
fig.suptitle('Best Macro F1 by Modality Count', fontsize=13, fontweight='bold', y=1.02)

for ax, cancer in zip(axes, TARGET_CANCERS):
    best_per_n = {}
    best_combo_per_n = {}
    for n in range(1, 7):
        combos_n = [c for c in all_combos if len(c) == n]
        f1s = {c: ablation_results[cancer][c] for c in combos_n}
        best = max(f1s, key=f1s.get)
        best_per_n[n] = f1s[best]
        best_combo_per_n[n] = '+'.join(MODALITY_NAMES[i] for i in best)

    ns  = list(best_per_n.keys())
    f1s = list(best_per_n.values())
    ax.plot(ns, f1s, 'o-', color='#2E86AB', linewidth=2, markersize=6, markerfacecolor='white', markeredgewidth=2)
    ax.set_title(CANCER_LABELS[cancer], fontweight='bold', fontsize=11)
    ax.set_xlabel('# Modalities', fontsize=9)
    if ax == axes[0]:
        ax.set_ylabel('Best Macro F1', fontsize=9)
    ax.set_xticks(range(1, 7))
    ax.set_ylim(0, 1.0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3)

    # Annotate best combo at each n
    for n, (f1, combo_name) in enumerate(zip(f1s, best_combo_per_n.values()), 1):
        short = combo_name if len(combo_name) < 20 else combo_name[:18]+'…'
        ax.annotate(short, (n, f1), textcoords='offset points', xytext=(4, 4),
                    fontsize=5.5, color='#444444', rotation=20)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig3_best_per_n_modalities.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved fig3_best_per_n_modalities.png")


# ── Figure 4: Modality importance — leave-one-out from full model ─────────────
full_combo = tuple(range(6))
fig, axes = plt.subplots(1, 5, figsize=(16, 4), sharey=False)
fig.suptitle('Modality Importance: Drop from Full Model (Macro F1 Delta)', fontsize=13, fontweight='bold', y=1.02)

for ax, cancer in zip(axes, TARGET_CANCERS):
    full_f1 = ablation_results[cancer][full_combo]
    deltas, colors = [], []
    for i, mod in enumerate(MODALITY_NAMES):
        loo_combo = tuple(j for j in range(6) if j != i)
        loo_f1    = ablation_results[cancer][loo_combo]
        delta     = full_f1 - loo_f1   # positive = modality helps
        deltas.append(delta)
        colors.append(MOD_COLORS[mod])

    bars = ax.bar(MODALITY_NAMES, deltas, color=colors, edgecolor='white', linewidth=0.5)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(CANCER_LABELS[cancer], fontweight='bold', fontsize=11)
    ax.set_ylabel('F1 Drop When Removed' if ax == axes[0] else '')
    ax.set_xticklabels(MODALITY_NAMES, rotation=45, ha='right', fontsize=9)
    for bar, val in zip(bars, deltas):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + (0.005 if val >= 0 else -0.015),
                f'{val:+.3f}', ha='center', va='bottom', fontsize=7.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig4_leave_one_out.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved fig4_leave_one_out.png")


# ── Summary table ─────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("ABLATION SUMMARY TABLE")
print("="*70)
print(f"{'Cancer':<14} {'Best Combo':<35} {'Best F1':>8} {'Full-6 F1':>10} {'Delta':>8}")
print("-"*70)
for cancer in TARGET_CANCERS:
    full_f1  = ablation_results[cancer][full_combo]
    best_combo = max(ablation_results[cancer], key=ablation_results[cancer].get)
    best_f1    = ablation_results[cancer][best_combo]
    best_name  = '+'.join(MODALITY_NAMES[i] for i in best_combo)
    delta      = best_f1 - full_f1
    print(f"{CANCER_LABELS[cancer]:<14} {best_name:<35} {best_f1:>8.4f} {full_f1:>10.4f} {delta:>+8.4f}")

print("\nSingle-modality rankings:")
for cancer in TARGET_CANCERS:
    singles = {MODALITY_NAMES[i]: ablation_results[cancer][(i,)] for i in range(6)}
    ranked  = sorted(singles.items(), key=lambda x: x[1], reverse=True)
    print(f"  {CANCER_LABELS[cancer]}: {' > '.join(f'{m}({f:.2f})' for m,f in ranked)}")

print(f"\nAll figures saved to {FIG_DIR}")

Saved fig1_single_modality.png
Saved fig2_full_heatmap.png
Saved fig3_best_per_n_modalities.png
Saved fig4_leave_one_out.png

ABLATION SUMMARY TABLE
Cancer         Best Combo                           Best F1  Full-6 F1    Delta
----------------------------------------------------------------------
BRCA           rna+meth+cnv+mut                      0.7513     0.7356  +0.0157
GBM/LGG        rna+meth+mirna                        0.8713     0.8217  +0.0496
UCEC           rna+meth+cnv+clin                     0.7634     0.6104  +0.1531
THCA           rna+meth+mut                          0.8171     0.7138  +0.1033
KIRC           rna+clin                              0.8722     0.4704  +0.4018

Single-modality rankings:
  BRCA: rna(0.68) > meth(0.64) > mirna(0.57) > clin(0.32) > cnv(0.30) > mut(0.13)
  GBM/LGG: meth(0.82) > mirna(0.57) > rna(0.54) > cnv(0.27) > clin(0.16) > mut(0.07)
  UCEC: rna(0.72) > meth(0.68) > mirna(0.55) > clin(0.33) > cnv(0.12) > mut(0.12)
  THCA: mirna(0.76) > rn

In [ ]:
# ── Unfrozen Encoder Fine-Tuning ─────────────────────────────────────────────
import copy
import pickle
from sklearn.metrics import f1_score, classification_report
from torch.utils.data import WeightedRandomSampler

def train_unfrozen_classifier(cancer, subtype_datasets, pretrained_model, mod_data, mod_maps,
                               epochs=40, lr_backbone=1e-4, lr_head=1e-3, batch_size=32):
    ds    = subtype_datasets[cancer]
    df    = ds['df']
    n_sub = ds['n_subtypes']
    y_all = df['subtype_label'].values
    cids  = df['uuid'].values
    idx   = np.arange(len(df))

    idx_tv, idx_test = train_test_split(idx, test_size=0.15, stratify=y_all, random_state=42)
    idx_train, idx_val = train_test_split(idx_tv, test_size=0.15/0.85,
                                           stratify=y_all[idx_tv], random_state=42)

    def make_loader(split_idx, weighted=False):
        labels = y_all[split_idx]
        dataset = SubtypeDataset(split_idx, labels, mod_data, mod_maps, cids[split_idx])
        sampler = None
        shuffle = True
        if weighted:
            counts  = np.bincount(labels)
            weights = 1.0 / counts[labels]
            sampler = WeightedRandomSampler(weights, len(weights))
            shuffle = False
        return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                          sampler=sampler, collate_fn=collate_fn)

    train_loader = make_loader(idx_train, weighted=True)
    val_loader   = make_loader(idx_val)
    test_loader  = make_loader(idx_test)

    # Deep copy the pretrained model so we don't overwrite it between cancers
    clf = copy.deepcopy(pretrained_model).to(DEVICE)

    # Replace the 33-class classifier with a fresh head for n_sub subtypes
    clf.classifier = nn.Sequential(
        nn.Linear(960, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.3),
        nn.Linear(256, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.2),
        nn.Linear(128, n_sub),
    ).to(DEVICE)

    # Separate LR for backbone (small) vs new head (normal)
    backbone_params = [p for n, p in clf.named_parameters() if not n.startswith('classifier')]
    head_params     = list(clf.classifier.parameters())
    opt = torch.optim.AdamW([
        {'params': backbone_params, 'lr': lr_backbone},
        {'params': head_params,     'lr': lr_head},
    ], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    best_val_f1, best_state = 0.0, None

    for epoch in range(1, epochs + 1):
        clf.train()
        for tensors, mask, labels in train_loader:
            tensors = [t.to(DEVICE) for t in tensors]
            mask, labels = mask.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad()
            logits, _, _ = clf(tensors, mask)
            nn.CrossEntropyLoss()(logits, labels).backward()
            torch.nn.utils.clip_grad_norm_(clf.parameters(), 1.0)
            opt.step()
        sched.step()

        clf.eval()
        preds, trues = [], []
        with torch.no_grad():
            for tensors, mask, labels in val_loader:
                tensors = [t.to(DEVICE) for t in tensors]
                mask = mask.to(DEVICE)
                logits, _, _ = clf(tensors, mask)
                preds.extend(logits.argmax(1).cpu().numpy())
                trues.extend(labels.numpy())
        val_f1 = f1_score(trues, preds, average='macro')
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state  = {k: v.clone() for k, v in clf.state_dict().items()}
        if epoch % 10 == 0:
            print(f"  [{cancer}] epoch {epoch:3d} | val macro-F1 {val_f1:.4f}")

    # Test
    clf.load_state_dict(best_state)
    clf.eval()
    preds, trues = [], []
    with torch.no_grad():
        for tensors, mask, labels in test_loader:
            tensors = [t.to(DEVICE) for t in tensors]
            mask = mask.to(DEVICE)
            logits, _, _ = clf(tensors, mask)
            preds.extend(logits.argmax(1).cpu().numpy())
            trues.extend(labels.numpy())

    test_f1  = f1_score(trues, preds, average='macro')
    test_acc = np.mean(np.array(preds) == np.array(trues))
    print(f"\n{'='*55}")
    print(f"{cancer} UNFROZEN — Test F1={test_f1:.4f}  Acc={test_acc:.4f}  (best val F1={best_val_f1:.4f})")
    print(f"{'='*55}")
    print(classification_report(trues, preds, target_names=ds['classes'], digits=3))

    return {'cancer': cancer, 'test_f1': test_f1, 'test_acc': test_acc,
            'best_val_f1': best_val_f1, 'preds': preds, 'trues': trues,
            'classes': ds['classes']}


# ── Run all 5 cancers ─────────────────────────────────────────────────────────
unfrozen_results = {}
for cancer in TARGET_CANCERS:
    print(f"\n{'='*55}\nUNFREEZING on {cancer}\n{'='*55}")
    unfrozen_results[cancer] = train_unfrozen_classifier(
        cancer, subtype_datasets, model, mod_data, mod_maps
    )

# Save
with open(PROC_DIR / 'subtypes' / 'unfrozen_results.pkl', 'wb') as f:
    pickle.dump(unfrozen_results, f)
print("\n✅ Unfrozen results saved")


# ── Compare frozen vs unfrozen ────────────────────────────────────────────────
print("\n" + "="*70)
print("FROZEN vs UNFROZEN COMPARISON")
print("="*70)
frozen_f1s = {
    'TCGA-GBM_LGG': 0.752,
    'TCGA-KIRC':    0.779,
    'TCGA-THCA':    0.715,
    'TCGA-BRCA':    0.681,
    'TCGA-UCEC':    0.629,
}
print(f"{'Cancer':<14} {'Frozen F1':>10} {'Unfrozen F1':>12} {'Delta':>8}")
print("-"*50)
for cancer in TARGET_CANCERS:
    frozen   = frozen_f1s[cancer]
    unfrozen = unfrozen_results[cancer]['test_f1']
    delta    = unfrozen - frozen
    print(f"{cancer:<14} {frozen:>10.4f} {unfrozen:>12.4f} {delta:>+8.4f}")


UNFREEZING on TCGA-BRCA
  [TCGA-BRCA] epoch  10 | val macro-F1 0.6608
  [TCGA-BRCA] epoch  20 | val macro-F1 0.6804
  [TCGA-BRCA] epoch  30 | val macro-F1 0.6898
  [TCGA-BRCA] epoch  40 | val macro-F1 0.6898

TCGA-BRCA UNFROZEN — Test F1=0.6754  Acc=0.7322  (best val F1=0.7060)
              precision    recall  f1-score   support

  BRCA.Basal      0.900     0.931     0.915        29
   BRCA.Her2      0.643     0.750     0.692        12
   BRCA.LumA      0.816     0.816     0.816        87
   BRCA.LumB      0.667     0.606     0.635        33
 BRCA.Normal      0.318     0.318     0.318        22

    accuracy                          0.732       183
   macro avg      0.669     0.684     0.675       183
weighted avg      0.731     0.732     0.731       183


UNFREEZING on TCGA-GBM_LGG
  [TCGA-GBM_LGG] epoch  10 | val macro-F1 0.9318
  [TCGA-GBM_LGG] epoch  20 | val macro-F1 0.9318
  [TCGA-GBM_LGG] epoch  30 | val macro-F1 0.9318
  [TCGA-GBM_LGG] epoch  40 | val macro-F1 0.9318

TCGA-G

In [ ]:
# ── RECOVERY CELL — run this first after every Colab restart ─────────────────
import os, time, pickle, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from itertools import combinations
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# ── Paths ─────────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

BASE_DIR  = Path('/content/drive/MyDrive/tcga_pancancer')
PROC_DIR  = BASE_DIR / 'processed'
MDL_DIR   = BASE_DIR / 'models'
RES_DIR   = BASE_DIR / 'results'
SUB_DIR   = PROC_DIR / 'subtypes'
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# ── Constants ─────────────────────────────────────────────────────────────────
MODALITY_NAMES = ['rna', 'meth', 'cnv', 'mirna', 'mut', 'clin']
TARGET_CANCERS = ['TCGA-BRCA', 'TCGA-GBM_LGG', 'TCGA-UCEC', 'TCGA-THCA', 'TCGA-KIRC']
LATENT_DIMS    = {'rna': 256, 'meth': 256, 'cnv': 128, 'mirna': 128, 'mut': 128, 'clin': 64}
HIDDEN_CFG     = {
    'rna':   [2048, 1024, 512],
    'meth':  [2048, 1024, 512],
    'cnv':   [256, 128],
    'mirna': [512, 256],
    'mut':   [512, 256],
    'clin':  [64, 64],
}

# ── Load modality matrices ────────────────────────────────────────────────────
print("Loading modality matrices...")
t0 = time.time()
mod_data, mod_maps, mod_dims = [], [], {}
for mod in MODALITY_NAMES:
    df = pd.read_parquet(PROC_DIR / f'{mod}_matrix.parquet')
    mod_maps.append({cid: i for i, cid in enumerate(df.index)})
    mod_data.append(df.values.astype(np.float32))
    mod_dims[mod] = df.shape[1]
    print(f"  {mod}: {df.shape}")
print(f"All matrices loaded in {time.time()-t0:.1f}s")

# ── Load pretrained pan-cancer model ─────────────────────────────────────────
# (only needed for frozen/unfrozen experiments, not ablation)
# model = torch.load(MDL_DIR / 'best_model.pt', map_location=DEVICE)
# model.eval()
# print("Pan-cancer model loaded")

# ── Load saved subtype datasets ───────────────────────────────────────────────
print("\nLoading subtype datasets...")
with open(SUB_DIR / 'subtype_datasets.pkl', 'rb') as f:
    subtype_datasets = pickle.load(f)
for cancer, ds in subtype_datasets.items():
    print(f"  {cancer}: {len(ds['df'])} samples, {ds['n_subtypes']} subtypes: {ds['classes']}")

# ── Load saved results (if they exist) ───────────────────────────────────────
ablation_results, unfrozen_results = {}, {}
abl_path = SUB_DIR / 'ablation_results.pkl'
unf_path = SUB_DIR / 'unfrozen_results.pkl'
if abl_path.exists():
    with open(abl_path, 'rb') as f:
        ablation_results = pickle.load(f)
    print(f"\nAblation results loaded: {sum(len(v) for v in ablation_results.values())} entries")
if unf_path.exists():
    with open(unf_path, 'rb') as f:
        unfrozen_results = pickle.load(f)
    print(f"Unfrozen results loaded: {len(unfrozen_results)} cancers")

# ── SubtypeDataset + collate_fn ───────────────────────────────────────────────
class SubtypeDataset(Dataset):
    def __init__(self, rows, labels, mod_data, mod_maps, case_ids_for_rows):
        self.rows     = rows
        self.labels   = labels
        self.mod_data = mod_data
        self.mod_maps = mod_maps
        self.cids     = case_ids_for_rows

    def __len__(self): return len(self.rows)

    def __getitem__(self, i):
        cid = self.cids[i]
        tensors, mask = [], []
        for data, mapping in zip(self.mod_data, self.mod_maps):
            if cid in mapping:
                tensors.append(torch.tensor(data[mapping[cid]], dtype=torch.float32))
                mask.append(1.0)
            else:
                tensors.append(torch.zeros(data.shape[1], dtype=torch.float32))
                mask.append(0.0)
        return tensors, torch.tensor(mask, dtype=torch.float32), \
               torch.tensor(self.labels[i], dtype=torch.long)

def collate_fn(batch):
    tensors_list, masks, labels = zip(*batch)
    return (
        [torch.stack([s[m] for s in tensors_list]) for m in range(len(tensors_list[0]))],
        torch.stack(masks),
        torch.stack(labels)
    )

# ── SubsetMultiModalModel ─────────────────────────────────────────────────────
def make_encoder(in_dim, latent_dim, hidden):
    layers, prev = [], in_dim
    for h in hidden:
        layers += [nn.Linear(prev, h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(0.2)]
        prev = h
    layers.append(nn.Linear(prev, latent_dim))
    return nn.Sequential(*layers)

class SubsetMultiModalModel(nn.Module):
    def __init__(self, active_indices, mod_dims, n_classes):
        super().__init__()
        self.active = list(active_indices)
        self.encoders = nn.ModuleList([
            make_encoder(
                mod_dims[MODALITY_NAMES[i]],
                LATENT_DIMS[MODALITY_NAMES[i]],
                HIDDEN_CFG[MODALITY_NAMES[i]]
            ) for i in self.active
        ])
        self.gate      = nn.Linear(len(self.active), len(self.active))
        fusion_dim     = sum(LATENT_DIMS[MODALITY_NAMES[i]] for i in self.active)
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, 128),        nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, n_classes),
        )

    def forward(self, tensors, mask):
        active_tensors = [tensors[i] for i in self.active]
        active_mask    = mask[:, self.active]
        zs    = [enc(x) for enc, x in zip(self.encoders, active_tensors)]
        gate  = torch.sigmoid(self.gate(active_mask))
        fused = torch.cat([z * gate[:, i:i+1] for i, z in enumerate(zs)], dim=-1)
        return self.classifier(fused)

print("\n✅ Recovery complete — SubtypeDataset, collate_fn, SubsetMultiModalModel all ready")
print(f"   mod_data: {len(mod_data)} modalities loaded")
print(f"   subtype_datasets: {list(subtype_datasets.keys())}")

Mounted at /content/drive
Device: cuda
Loading modality matrices...
  rna: (11505, 5000)
  meth: (12527, 10000)
  cnv: (15000, 41)
  mirna: (11441, 508)
  mut: (9106, 501)
  clin: (11428, 7)
All matrices loaded in 54.9s

Loading subtype datasets...
  TCGA-BRCA: 1216 samples, 5 subtypes: ['BRCA.Basal', 'BRCA.Her2', 'BRCA.LumA', 'BRCA.LumB', 'BRCA.Normal']
  TCGA-GBM_LGG: 723 samples, 7 subtypes: ['GBM_LGG.Classic-like', 'GBM_LGG.Codel', 'GBM_LGG.G-CIMP-high', 'GBM_LGG.G-CIMP-low', 'GBM_LGG.LGm6-GBM', 'GBM_LGG.Mesenchymal-like', 'GBM_LGG.PA-like']
  TCGA-UCEC: 507 samples, 4 subtypes: ['UCEC.CN_HIGH', 'UCEC.CN_LOW', 'UCEC.MSI', 'UCEC.POLE']
  TCGA-THCA: 482 samples, 5 subtypes: ['THCA.1', 'THCA.2', 'THCA.3', 'THCA.4', 'THCA.5']
  TCGA-KIRC: 417 samples, 4 subtypes: ['KIRC.1', 'KIRC.2', 'KIRC.3', 'KIRC.4']

Ablation results loaded: 315 entries
Unfrozen results loaded: 5 cancers

✅ Recovery complete — SubtypeDataset, collate_fn, SubsetMultiModalModel all ready
   mod_data: 6 modalities loa

In [ ]:
# ── Phase 1: Multi-Seed + Cross-Validation Ablation (Colab version) ──────────
# 315 combos × 5 seeds × 5 folds = 7,875 runs
# Resume-safe: skips completed runs on restart
# Run the recovery cell first, then run this cell

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
import json, time, copy
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────────
SEEDS   = [42, 123, 456, 789, 2026]
N_FOLDS = 5
EPOCHS  = 60
LR      = 1e-3

# Results saved here — one JSON per run, survives disconnects
PHASE1_DIR = SUB_DIR / 'phase1_results'
PHASE1_DIR.mkdir(exist_ok=True)

def get_result_path(cancer, combo, seed, fold):
    combo_str = '+'.join(MODALITY_NAMES[i] for i in combo)
    fname = f"{cancer.replace('TCGA-','')}_{combo_str}_s{seed}_f{fold}.json"
    return PHASE1_DIR / fname

def get_fold_indices(y_all, seed, fold):
    y_arr = np.array(y_all)
    skf   = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    splits = list(skf.split(np.zeros(len(y_arr)), y_arr))
    train_val_idx, test_idx = splits[fold]
    train_idx, val_idx = train_test_split(
        train_val_idx, test_size=0.15,
        stratify=y_arr[train_val_idx], random_state=seed
    )
    return train_idx, val_idx, test_idx

def train_one_run(cancer, combo, seed, fold):
    """Train one (combo × cancer × seed × fold). Returns result dict."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    ds     = subtype_datasets[cancer]
    df     = ds['df']
    n_sub  = ds['n_subtypes']
    y_all  = df['subtype_label'].values
    cids   = df['uuid'].values

    train_idx, val_idx, test_idx = get_fold_indices(y_all.tolist(), seed, fold)

    def make_loader(idx, weighted=False):
        labels  = y_all[idx]
        dataset = SubtypeDataset(idx, labels, mod_data, mod_maps, cids[idx])
        sampler, shuffle = None, True
        if weighted:
            counts  = np.bincount(labels, minlength=n_sub)
            counts  = np.where(counts == 0, 1, counts)
            weights = 1.0 / counts[labels]
            sampler = WeightedRandomSampler(weights, len(weights))
            shuffle = False
        return DataLoader(dataset, batch_size=32, shuffle=shuffle,
                          sampler=sampler, collate_fn=collate_fn)

    train_loader = make_loader(train_idx, weighted=True)
    val_loader   = make_loader(val_idx)
    test_loader  = make_loader(test_idx)

    model = SubsetMultiModalModel(list(combo), mod_dims, n_sub).to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    best_val_f1, best_state = 0.0, None
    for epoch in range(EPOCHS):
        model.train()
        for tensors, mask, labels in train_loader:
            tensors = [t.to(DEVICE) for t in tensors]
            mask, labels = mask.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad()
            nn.CrossEntropyLoss()(model(tensors, mask), labels).backward()
            opt.step()
        sched.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for tensors, mask, labels in val_loader:
                tensors = [t.to(DEVICE) for t in tensors]
                preds.extend(model(tensors, mask.to(DEVICE)).argmax(1).cpu().numpy())
                trues.extend(labels.numpy())
        val_f1 = f1_score(trues, preds, average='macro', zero_division=0)
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state  = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for tensors, mask, labels in test_loader:
            tensors = [t.to(DEVICE) for t in tensors]
            preds.extend(model(tensors, mask.to(DEVICE)).argmax(1).cpu().numpy())
            trues.extend(labels.numpy())

    test_f1  = float(f1_score(trues, preds, average='macro', zero_division=0))
    test_acc = float(np.mean(np.array(preds) == np.array(trues)))

    return {
        'cancer':      cancer,
        'combo':       list(combo),
        'combo_name':  '+'.join(MODALITY_NAMES[i] for i in combo),
        'seed':        seed,
        'fold':        fold,
        'test_f1':     test_f1,
        'test_acc':    test_acc,
        'best_val_f1': float(best_val_f1),
        'n_train':     int(len(train_idx)),
        'n_val':       int(len(val_idx)),
        'n_test':      int(len(test_idx)),
    }


# ── Main loop ─────────────────────────────────────────────────────────────────
all_combos = []
for r in range(1, 7):
    all_combos.extend(combinations(range(6), r))

total     = len(TARGET_CANCERS) * len(all_combos) * len(SEEDS) * N_FOLDS
completed = len(list(PHASE1_DIR.glob('*.json')))
print(f"Total runs: {total}")
print(f"Already completed: {completed}")
print(f"Remaining: {total - completed}")
print(f"Estimated time remaining: {(total - completed) * 25 / 3600:.1f} hrs on A100\n")

run_count = 0
t_session = time.time()

for cancer in TARGET_CANCERS:
    for combo in all_combos:
        for seed in SEEDS:
            for fold in range(N_FOLDS):

                # Skip if already done
                rpath = get_result_path(cancer, combo, seed, fold)
                if rpath.exists():
                    continue

                t0 = time.time()
                result = train_one_run(cancer, combo, seed, fold)
                elapsed = time.time() - t0

                # Save immediately
                with open(rpath, 'w') as f:
                    json.dump(result, f)

                run_count += 1
                completed += 1

                # Progress print every run
                combo_name = result['combo_name']
                print(f"[{completed}/{total}] {cancer.replace('TCGA-','')} "
                      f"{combo_name:<30} s{seed} f{fold} "
                      f"F1={result['test_f1']:.4f} ({elapsed:.0f}s) "
                      f"session_total={run_count}")

print(f"\n✅ Session complete. {run_count} new runs. {completed}/{total} total.")

Total runs: 7875
Already completed: 2233
Remaining: 5642
Estimated time remaining: 39.2 hrs on A100

[2234/7875] GBM_LGG rna+cnv+mut                    s123 f3 F1=0.5905 (24s) session_total=1
[2235/7875] GBM_LGG rna+cnv+mut                    s123 f4 F1=0.7170 (18s) session_total=2
[2236/7875] GBM_LGG rna+cnv+mut                    s456 f0 F1=0.6083 (17s) session_total=3
[2237/7875] GBM_LGG rna+cnv+mut                    s456 f1 F1=0.5250 (17s) session_total=4
[2238/7875] GBM_LGG rna+cnv+mut                    s456 f2 F1=0.5592 (17s) session_total=5
[2239/7875] GBM_LGG rna+cnv+mut                    s456 f3 F1=0.5510 (18s) session_total=6
[2240/7875] GBM_LGG rna+cnv+mut                    s456 f4 F1=0.4894 (17s) session_total=7
[2241/7875] GBM_LGG rna+cnv+mut                    s789 f0 F1=0.5750 (17s) session_total=8
[2242/7875] GBM_LGG rna+cnv+mut                    s789 f1 F1=0.5478 (16s) session_total=9
[2243/7875] GBM_LGG rna+cnv+mut                    s789 f2 F1=0.5704 (17s) sessi

In [ ]:
PHASE1_DIR = SUB_DIR / 'phase1_results'

result_files = list(PHASE1_DIR.glob('*.json'))
rows = []
for f in result_files:
    with open(f) as fp:
        rows.append(json.load(fp))
df_check = pd.DataFrame(rows)

mask = df_check['seed'].isin([42,123,456]) & df_check['fold'].isin([0,1,2])
print(df_check[mask].groupby('cancer').size())
print(f"\nTotal 3x3 runs done: {mask.sum()} / {315*5*9}")
print(f"Remaining: {315*5*9 - mask.sum()}")

cancer
TCGA-BRCA       567
TCGA-GBM_LGG    432
dtype: int64

Total 3x3 runs done: 999 / 14175
Remaining: 13176


In [ ]:
# Aggregate whatever you have right now, all seeds/folds
PHASE1_DIR = SUB_DIR / 'phase1_results'

result_files = list(PHASE1_DIR.glob('*.json'))
rows = []
for f in result_files:
    with open(f) as fp:
        rows.append(json.load(fp))
df_all = pd.DataFrame(rows)

print(f"Total runs: {len(df_all)}")
print("\nRuns per cancer:")
print(df_all.groupby('cancer').size())
print("\nRuns per (cancer, combo) — min/mean/max:")
print(df_all.groupby(['cancer','combo_name']).size().describe())

Total runs: 2767

Runs per cancer:
cancer
TCGA-BRCA       1575
TCGA-GBM_LGG    1192
dtype: int64

Runs per (cancer, combo) — min/mean/max:
count    111.000000
mean      24.927928
std        0.759326
min       17.000000
25%       25.000000
50%       25.000000
75%       25.000000
max       25.000000
dtype: float64
